# Ration Early or Run Dry?

## A reservoir trade-off in four TaqSim blocks

Dr. Tobias Siegfried, hydrosolutions GmbH · CC BY 4.0

<p><img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"></p>
<p><b>hydrosolutions GmbH</b> · Venusstrasse 29, 8050 Zürich, Switzerland · <a href="https://www.hydrosolutions.ch">www.hydrosolutions.ch</a> · hs@hydrosolutions.ch · +41 43 535 05 80</p>

# Вопрос

Водохранилище наполняется зимой. Летом из него берёт воду хозяйство. В большинстве лет воды хватает. В сухой год
её не хватает, и у оператора есть два способа управлять водохранилищем:

- **Подавать полностью, пока водохранилище не опустеет.** Хозяйство получает всё, что просит, так долго, как
  возможно. Если вода кончается, последние недели лета остаются почти ни с чем.
- **Лимитировать заранее.** Когда воды в водохранилище становится мало, подавать лишь часть того, что просит
  хозяйство. Воды хватает дольше, но дефицит случается чаще, в том числе в годы, когда воды и так хватило бы.

Что лучше? Это зависит от того, что считать. В этом блокноте собрана самая маленькая модель, которая это
показывает: четыре блока TaqSim, два рычага, две цели. Затем поиск поручается оптимизатору.

**Что вы вынесете из блокнота**

1. Как собрать и запустить небольшую водную систему в TaqSim, блок за блоком.
2. Почему две разумные цели могут тянуть в разные стороны и как увидеть все разумные компромиссы сразу.
3. Как поручить поиск оптимизатору и как читать то, что он возвращает.

> **Термины.** Инженеры называют «подавать полностью, пока не опустеет» *стандартной политикой управления*, а
> «лимитировать заранее» — *хеджированием*. План, который ни один другой план не превосходит по всем целям,
> называется *Парето-оптимальным*, а множество таких планов — *фронтом Парето*. В этом блокноте они называются
> «непревзойдённые планы».

# Система из четырёх блоков

TaqSim описывает водную систему как **сеть**: узлы, которые хранят, используют или передают воду, соединённые
рёбрами, по которым вода течёт. В нашей сети будет четыре узла и три ребра. Начнём с трёх, реки, хозяйства и
нижнего бьефа, и добавим водохранилище, когда увидим, что без него идёт не так.

In [ ]:
# @title Настройка: сначала запустите эту ячейку
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# A runtime that ran this cell earlier may hold an older copy of the course: bring the folder up to date and
# forget any course module Python has already loaded, so that the current code is what gets imported.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
# @title Код: setup
#| label: setup
# Standard tools
import inspect  # to show the source code of a function or class inside this notebook
import logging

import pandas as pd  # tables
from IPython.display import HTML, Markdown  # to show formatted text and HTML as cell output

# TaqSim itself: the four kinds of node we use, the edge that joins them, and the system that holds them
from taqsim import Demand, Edge, Sink, Source, Storage, TimeSeries, WaterSystem
from taqsim.node import NoLoss  # a loss rule that loses nothing (reservoirs need one)
from taqsim.node.events import DeficitRecorded, WaterSpilled  # the events a short Demand and a full Storage record
from taqsim.time import Frequency  # the length of one simulation step (monthly here)

# The helpers written for this coursebook (src/zarafshan_taqsim/rationing_coursebook.py)
from zarafshan_taqsim.rationing_coursebook import (
    CAPACITY_MM3,
    EVAPORATION_SHARE_EXAMPLE,
    MONTHLY_NEED_MM3,
    NO_RATIONING,
    PAPER_REGIME,
    START_STORAGE_MM3,
    SUMMER_NEED_MM3,
    WINTER_NEED_MM3,
    YEARS,
    Plan,
    RationingRelease,
    SUMMER_PLAN_FOR_WINTER,
    WINTER_FIELDS_TOTAL_MM3,
    WINTER_IN_FULL,
    SummerEvaporation,
    WinterPlan,
    WinterRelease,
    configure,
    drought_window_start,
    explorer_html,
    explorer_payload,
    front_of,
    objectives,
    optimise_plans,
    plan_grid,
    forecast_value,
    least_winter_shortage,
    plot_forecast_skill,
    plot_forecast_value,
    plot_front_shift,
    plot_inflows,
    plot_network,
    plot_regime_comparison,
    plot_runs,
    plot_score_maps,
    plot_shortage_years,
    plot_system_sketch,
    plot_tradeoff,
    plot_winter_decisions,
    plot_winter_fronts,
    score_by_skill,
    score_plans,
    score_winter_plans,
    season_statistics,
    season_totals,
    seasonal_forecast,
    simulate_plan,
    simulate_winter,
    simulate_without_reservoir,
    synthetic_inflows,
    winter_grid,
    winter_plan_within,
)

logging.basicConfig(level=logging.WARNING)  # only show warnings and errors, not progress messages
pd.set_option("display.precision", 2)  # two decimals in tables

LANG = "ru"  # the web page shows all three languages and lets you switch; a notebook is built for one language


def say(en: str, uz: str, ru: str) -> Markdown:
    """Text in three languages: all three on the web page (the switcher picks one), one in a notebook."""
    if LANG:
        return Markdown({"en": en, "uz": uz, "ru": ru}[LANG])
    blocks = (("en", en), ("uz", uz), ("ru", ru))
    return Markdown("\n\n".join(f"::: {{.lang-{code}}}\n{text}\n:::" for code, text in blocks))


def show_source(thing) -> Markdown:
    """Show the source code of a function or class as a code block."""
    return Markdown(f"```python\n{inspect.getsource(thing)}```")

In [ ]:
# @title Код: fig-network
#| label: fig-network
#| fig-cap: "The model as a network. Each node is one TaqSim block; each edge is named after the nodes it joins."
#| echo: false
_ = plot_network()

- **river** (река) — `Source`: каждый месяц подаёт воду в сеть по списку чисел.
- **reservoir** (водохранилище) — `Storage`: хранит воду до своей ёмкости, часть теряет (если мы так зададим) и
  часть сбрасывает по правилу. Здесь принимаются решения.
- **farm** (хозяйство) — `Demand`: каждый месяц запрашивает заданный объём и записывает, сколько не получило.
- **downstream** (нижний бьеф) — `Sink`: всё, что хозяйство не использовало, покидает сеть здесь.

In [ ]:
# @title Код: fig-system
#| label: fig-system
#| fig-cap: "The same four blocks with their numbers. Water moves from left to right."
#| echo: false
_ = plot_system_sketch()

Числа взяты из классической статьи о надёжности водохранилищ (Hashimoto, Stedinger и Loucks, 1982, с. 17 и
таблица 1) с одним изменением. Река в статье многоводна зимой. Реки Центральной Азии питаются талыми водами снега и
ледников: пик стока приходится на лето, а зимой сток минимален. Поэтому мы меняем местами два сезона из статьи, а
всё остальное оставляем. [Раздел сравнения](#sec-paper-regime) прогоняет реку из статьи рядом с нашей. Объёмы в
миллионах кубометров (млн м³).

| Величина | Значение |
|---|---|
| Ёмкость водохранилища | 40 млн м³ |
| Потребность хозяйства летом | 45 млн м³ (7,5 млн м³ в месяц в течение шести месяцев) |
| Что нужно хозяйству зимой | 5 млн м³ (0,83 млн м³ в месяц в течение шести месяцев): небольшая потребность вне сезона, взятая из статьи |
| Приток зимой | в среднем 25 млн м³, стандартное отклонение 10 млн м³ (в статье: 40 и 15) |
| Приток летом | в среднем 40 млн м³, стандартное отклонение 15 млн м³ (в статье: 25 и 10) |

Летняя потребность (45) больше среднего летнего притока (40), поэтому хозяйство зависит от воды, накопленной
зимой. Водохранилище (40) меньше потребности, а наполнить его можно только из скудного зимнего стока, так что
запаса на ошибку почти нет.

Модельный год — шесть зимних месяцев, затем шесть летних; календарные даты не заданы. Читайте его как
гидрологический год, принятый в бассейне Зарафшана: зима — с октября по март, лето (вегетационный период) — с
апреля по сентябрь. Внутри сезона от месяца к месяцу ничего не меняется: приток каждого сезона распределён
равномерно по его шести месяцам, как и потребность хозяйства. Единственная сезонность — зима против лета, как в
статье. Настоящий месячный ход Зарафшана с июльским пиком показан в игре «Зарафшан в шестнадцати блоках» и в
учебнике по моделированию.

**Эти числа — настройки, а не константы.** Ячейка ниже задаёт их для этой тетради. Измените значение, запустите
тетрадь заново отсюда, и каждый расчёт, таблица и рисунок последуют за ним. В тексте приводятся значения по
умолчанию; у примерных планов порог 16 млн м³, поэтому оставьте ёмкость выше него.

In [ ]:
# @title Код: settings
#| label: settings
settings = configure(
    capacity=40.0,  # the reservoir, Mm³
    summer_need=45.0,  # what the farm needs over the six summer months, Mm³
    winter_need=5.0,  # and over the six winter months
    winter_inflow=(25.0, 10.0),  # winter inflow: mean and standard deviation, Mm³ per season
    summer_inflow=(40.0, 15.0),  # summer inflow: mean and standard deviation
    start_storage=20.0,  # water in the reservoir when a simulation starts, Mm³
    years=100,  # length of the synthetic river
    seed=72,  # the random seed of the river: another seed, another river with the same statistics
)
# The names the text uses follow the settings
from zarafshan_taqsim.rationing_coursebook import CAPACITY_MM3, MONTHLY_NEED_MM3, START_STORAGE_MM3, SUMMER_NEED_MM3, WINTER_NEED_MM3, YEARS

settings.style.hide(axis="index")

## Река

Ста лет измерений у нас нет, поэтому мы создаём реку сами: сто лет зимних и летних притоков со средними и
разбросом из таблицы выше. За сухим сезоном, как и в настоящих реках, чаще следует ещё один сухой сезон.

In [ ]:
# @title Код: river
#| label: river
# The river is drawn from the settings cell: its winter and summer inflow (mean and standard deviation),
# its years and its seed. 100 years x 12 months = 1 200 numbers, each a month's inflow in Mm³.
inflows = synthetic_inflows()

# Check the river against the numbers we asked for: averages, spread and the link from one season to the next
season_statistics(inflows).style.hide(axis="index").format(precision=2)

`synthetic_inflows()` всегда даёт одну и ту же реку (используется фиксированное зерно генератора случайных
чисел), поэтому любой результат в этом блокноте воспроизводим. Таблица сравнивает статистику реки с заданными
значениями; они совпадают с точностью до нескольких процентов.

In [ ]:
# @title Код: fig-river
#| label: fig-river
#| fig-cap: "One hundred synthetic years. In many years the summer inflow is far below what the farm needs."
#| echo: false
_ = plot_inflows(inflows)

> **Чем это отличается от статьи.** (1) Сезоны поменяны местами, как объяснено выше. (2) Приток каждого сезона
> равномерно распределён по шести месячным шагам. (3) В статье правила управления выводятся математически; мы
> используем простое правило с двумя рычагами. (4) Связь между сезонами задана для логарифмов стока — это
> допущение. Поэтому числа в этом блокноте наши собственные, а не воспроизведение чисел статьи.

# До водохранилища: река и хозяйство

Начнём без водохранилища. Хозяйство стоит на реке и берёт то, что река приносит; что оно не использует, течёт
дальше. Три блока, два ребра и никаких решений: правила нет, потому что нечего хранить.

In [ ]:
# @title Код: fig-river-farm
#| label: fig-river-farm
#| fig-cap: "The system without a reservoir: three blocks, two edges."
#| echo: false
_ = plot_network(reservoir=False)

Сборка модели TaqSim всегда состоит из одних и тех же четырёх шагов: создать пустую систему, добавить узлы,
добавить рёбра, проверить и запустить. Вот все четыре для трёх блоков.

In [ ]:
# @title Код: river-farm
#| label: river-farm
# 1. An empty system. One simulation step is one month.
river_farm = WaterSystem(frequency=Frequency.MONTHLY)

# 2. Three nodes: the river gives water, the farm asks for it, the outlet takes what is left.
river_farm.add_node(Source(id="river", inflow=TimeSeries(values=list(inflows))))
river_farm.add_node(Demand(id="farm", requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS)))
river_farm.add_node(Sink(id="downstream"))

# 3. Two edges, each named after the nodes it joins.
for source, target in [("river", "farm"), ("farm", "downstream")]:
    river_farm.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check the network, then run all 1 200 months.
river_farm.validate()
river_farm.simulate(len(inflows))

# The farm records a DeficitRecorded event in every month it gets less than it needs; add up the missing water
not_delivered = sum(event.deficit for event in river_farm.nodes["farm"].events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.1f} Mm³ per year, on average")

In [ ]:
# @title Код: fig-why-a-reservoir
#| label: fig-why-a-reservoir
#| fig-cap: "The worst month of each year without a reservoir (top) and with a reservoir of 40 Mm³ that delivers in full until it is empty (bottom)."
#| echo: false
without_reservoir = simulate_without_reservoir(inflows)  # the three blocks above, run by a helper
with_reservoir = simulate_plan(inflows, NO_RATIONING)  # the four blocks built in the next section
_ = plot_shortage_years(
    {
        "River and farm, no reservoir": without_reservoir,
        "With a reservoir of 40 Mm³, delivered in full until empty": with_reservoir,
    }
)

In [ ]:
# @title Код: why-a-reservoir
#| label: why-a-reservoir
#| echo: false
a, b = without_reservoir.scores, with_reservoir.scores
say(
    f"That is a lot: the farm asks for {SUMMER_NEED_MM3 + WINTER_NEED_MM3:g} Mm³ a year and misses "
    f"**{a.mean_shortage:.1f}** of them, and it is short in **{a.years_short} of {YEARS} years**. The reason is not "
    f"a lack of water but its timing: the farm needs {SUMMER_NEED_MM3:g} Mm³ in summer, the river brings 40 on "
    f"average, and the 25 Mm³ that arrive in winter flow past the farm unused. The lower panel shows what a "
    f"reservoir does. A reservoir of {CAPACITY_MM3:g} Mm³ carries winter water into summer: the short years fall "
    f"from {a.years_short} to **{b.years_short}**, the water missed from {a.mean_shortage:.1f} to "
    f"**{b.mean_shortage:.1f} Mm³ a year**. This is what reservoirs are for.\n\n"
    f"Two things it does not do. It does not change the single worst month ({a.worst_month:.0f} % short in both "
    f"panels): in the driest summer the reservoir is already empty, because it holds only {CAPACITY_MM3:g} Mm³ and "
    f"can only fill from the low winter flow. And it brings a decision with it, how to release the water it holds, "
    f"which the river alone never asked of us. The rest of this coursebook is about that decision.",
    f"Bu koʻp: xoʻjalik yiliga {SUMMER_NEED_MM3 + WINTER_NEED_MM3:g} mln m³ soʻraydi va shundan "
    f"**{a.mean_shortage:.1f}** ini olmaydi, {YEARS} yildan **{a.years_short} tasida** taqchillik koʻradi. Sabab suv "
    f"yetishmasligida emas, uning vaqtida: xoʻjalikka yozda {SUMMER_NEED_MM3:g} mln m³ kerak, daryo oʻrtacha 40 "
    f"keltiradi, qishda kelgan 25 mln m³ esa xoʻjalik yonidan foydalanilmay oqib oʻtadi. Pastki panel suv ombori "
    f"nima qilishini koʻrsatadi. {CAPACITY_MM3:g} mln m³ li suv ombori qishki suvni yozga olib oʻtadi: taqchil yillar "
    f"{a.years_short} tadan **{b.years_short} taga**, olinmagan suv yiliga {a.mean_shortage:.1f} dan "
    f"**{b.mean_shortage:.1f} mln m³ ga** tushadi. Suv omborlari ana shu uchun quriladi.\n\n"
    f"U qilmaydigan ikki narsa bor. U eng yomon oyni oʻzgartirmaydi (ikkala panelda ham {a.worst_month:.0f} % "
    f"taqchillik): eng qurgʻoqchil yozda ombor allaqachon boʻsh, chunki u bor-yoʻgʻi {CAPACITY_MM3:g} mln m³ sigʻdiradi "
    f"va faqat kam suvli qishki oqimdan toʻlishi mumkin. Va u oʻzi bilan bir qarorni olib keladi: saqlagan suvini "
    f"qanday chiqarish kerak, buni daryoning oʻzi hech qachon bizdan soʻramagan edi. Bu darslikning qolgan qismi ana "
    f"shu qaror haqida.",
    f"Это много: хозяйство просит {SUMMER_NEED_MM3 + WINTER_NEED_MM3:g} млн м³ в год и недополучает "
    f"**{a.mean_shortage:.1f}** из них, а дефицит испытывает в **{a.years_short} из {YEARS} лет**. Причина не в "
    f"нехватке воды, а в её сроках: хозяйству нужно {SUMMER_NEED_MM3:g} млн м³ летом, река приносит в среднем 40, а "
    f"25 млн м³, приходящие зимой, протекают мимо хозяйства без пользы. Нижняя панель показывает, что делает "
    f"водохранилище. Водохранилище на {CAPACITY_MM3:g} млн м³ переносит зимнюю воду в лето: число дефицитных лет "
    f"падает с {a.years_short} до **{b.years_short}**, недополученная вода с {a.mean_shortage:.1f} до "
    f"**{b.mean_shortage:.1f} млн м³ в год**. Для этого водохранилища и строят.\n\n"
    f"Двух вещей оно не делает. Оно не меняет самый худший месяц ({a.worst_month:.0f} % дефицита на обеих панелях): в "
    f"самое сухое лето водохранилище уже пусто, потому что вмещает лишь {CAPACITY_MM3:g} млн м³ и наполняется только "
    f"из скудного зимнего стока. И оно приносит с собой решение, как выпускать накопленную воду, которого река сама "
    f"по себе от нас никогда не требовала. Остальная часть этого учебника об этом решении.",
)

# Собираем в TaqSim

## Правило с двумя рычагами

**Что такое класс, простыми словами.** Программа вроде TaqSim собрана из *объектов*: водохранилище, хозяйство,
правило. **Класс** описывает один вид объекта: что он помнит и что умеет делать. Представьте печатный бланк с
несколькими полями и короткой инструкцией на обороте. Класс — это чистый бланк; каждый объект, созданный из него, —
один заполненный экземпляр. Наш класс правила помнит два числа, порог и долю подачи (его *поля*), и умеет делать одно,
`release(...)`, о чём TaqSim просит его каждый месяц (его *метод*). Запись `RationingRelease(trigger=16, ration=0.8)`
заполняет один экземпляр бланка. `Plan`, `Storage` и `Demand` ниже — другие бланки того же рода. Вот и всё, что здесь
значит «объектно-ориентированный»: модель — это несколько таких бланков, передающих друг другу воду. Больше ничего не
нужно, чтобы читать или менять эту тетрадь.

Водохранилищу в TaqSim нужно *правило сброса*: небольшой класс, который говорит, сколько воды выпускать каждый
месяц. У нашего правила два рычага.

- **Порог** (trigger, млн м³): лимитирование начинается, когда в водохранилище меньше этого объёма.
- **Доля** (ration, от 0 до 1): какая часть летней потребности тогда подаётся.

In [ ]:
# @title Код: rule-source
#| label: rule-source
#| echo: false
show_source(RationingRelease)

Читаем класс сверху вниз:

- `@dataclass(frozen=True)` делает его маленькой неизменяемой записью: в ней два рычага и больше ничего.
- `__params__` называет рычаги, которые может крутить оптимизатор, а `__bounds__` — в каких пределах.
- `trigger` и `ration` — сами рычаги со значениями по умолчанию (0 и 1: без лимитирования).
- `release(...)` TaqSim вызывает раз в месяц. Метод получает водохранилище (`node`, из которого читает текущий
  объём), приток этого месяца и шаг времени `t` и возвращает объём сброса в млн м³.

Порог 0 (или доля 1) означает полное отсутствие лимитирования.

## Соединяем четыре блока

Те же четыре шага, что и для трёх блоков: создать пустую систему, добавить узлы, добавить рёбра, проверить и
запустить. Водохранилище встаёт между рекой и хозяйством, и ему нужно правило из раздела выше.

In [ ]:
# @title Код: wire
#| label: wire
# 1. An empty system. One simulation step is one month.
system = WaterSystem(frequency=Frequency.MONTHLY)

# 2. The four nodes. Each has an id (its name in the network) and the settings its kind needs.
system.add_node(
    Source(
        id="river",
        inflow=TimeSeries(values=list(inflows)),  # one value per month, Mm³
    )
)
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 40 Mm³; more than this spills
        initial_storage=START_STORAGE_MM3,  # 20 Mm³ in store when the simulation starts
        release_policy=RationingRelease(trigger=0.0, ration=1.0),  # our rule; these knobs mean no rationing
        loss_rule=NoLoss(),  # no evaporation for now
    )
)
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS),  # the 12 monthly needs, repeated 100 times
    )
)
system.add_node(Sink(id="downstream"))

# 3. The three edges. An edge is named after the two nodes it joins and carries whatever its source lets out.
for source, target in [("river", "reservoir"), ("reservoir", "farm"), ("farm", "downstream")]:
    system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check that the network is complete (every node connected, every setting valid), then run all 1 200 months.
system.validate()
system.simulate(len(inflows))

Запуск системы ничего не печатает. Вместо этого каждый узел ведёт журнал **событий**: что он получил, сохранил,
сбросил, потерял или не смог подать, месяц за месяцем. Хозяйство записывает событие `DeficitRecorded` в каждом
месяце, когда получает меньше, чем нужно, и каждое такое событие хранит недостающий объём:

In [ ]:
# @title Код: first-read
#| label: first-read
farm = system.nodes["farm"]  # look up a node by its id

# Add up the missing volume over all deficit events, then divide by the number of years
not_delivered = sum(event.deficit for event in farm.events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.2f} Mm³ per year, on average")

Далее `simulate_plan(inflows, plan)` выполняет ровно эти шаги (собрать, проверить, рассчитать, прочитать события)
и возвращает результат в удобном виде: объём в водохранилище по месяцам, дефицит по месяцам и три показателя,
которые вводятся ниже.

# Первый расчёт: подавать полностью, пока не опустеет

In [ ]:
# @title Код: fig-first-run
#| label: fig-first-run
#| fig-cap: "No rationing. Left: water in the reservoir over twenty of the hundred years. Right: for each year, how short the worst month was."
no_rationing = simulate_plan(inflows, NO_RATIONING)  # NO_RATIONING is Plan(trigger=0, ration=1)
first_year = drought_window_start(no_rationing)  # a 20-year window that contains the worst drought

# One row per plan; the right panel shows the 20 years starting in first_year, shaded in the left panel
_ = plot_runs({"Deliver in full until empty": no_rationing}, first_year=first_year)

Левая панель показывает все 100 лет. Для каждого года один столбик: худший месяц этого года, то есть доля
потребности этого месяца, которую хозяйство не получило. Столбик 72 % означает, что в худшем месяце хозяйство
получило лишь 28 % того, что просило; в большинстве лет столбика нет вовсе. Правая панель показывает крупно
выделенные 20 лет вокруг самой сильной засухи: водохранилище наполняется каждую зиму (восходящие линии) и
срабатывается каждое лето, а в самые сухие годы опустошается до нуля.

План оцениваем тремя числами:

- **Средний дефицит**: вода, которая была нужна хозяйству, но не подана, в млн м³ в год.
- **Худший месяц**: доля потребности, не поданная хозяйству в самом худшем месяце за сто лет.
- **Дефицитные годы**: число лет хотя бы с одним дефицитным месяцем.

In [ ]:
# @title Код: first-scores
#| label: first-scores
#| echo: false
s = no_rationing.scores
say(
    f"Without rationing the farm is short in **{s.years_short} of 100 years**. On average **{s.mean_shortage:.1f} Mm³ "
    f"per year** is not delivered, which is little. But when it goes wrong, it goes badly wrong: in the worst month "
    f"the farm gets **{100 - s.worst_month:.0f} %** of what it needs ({s.worst_month:.0f} % short).",
    f"Cheklashsiz xoʻjalik **100 yildan {s.years_short} tasida** taqchillik koʻradi. Oʻrtacha **yiliga "
    f"{s.mean_shortage:.1f} mln m³** yetkazib berilmaydi, bu kam. Lekin ishkal chiqqanda, qattiq chiqadi: eng yomon "
    f"oyda xoʻjalik keragining **{100 - s.worst_month:.0f} %** ini oladi ({s.worst_month:.0f} % taqchillik).",
    f"Без лимитирования хозяйство испытывает дефицит в **{s.years_short} из 100 лет**. В среднем не подаётся "
    f"**{s.mean_shortage:.1f} млн м³ в год** — немного. Но когда дело идёт плохо, оно идёт очень плохо: в худшем "
    f"месяце хозяйство получает **{100 - s.worst_month:.0f} %** потребности (дефицит {s.worst_month:.0f} %).",
)

**Куда уходит вода, когда водохранилище полно?** Она сбрасывается. Блок `Storage` пропускает всё, что не
помещается, и записывает это как событие `WaterSpilled`; сброс течёт к хозяйству так же, как попуск, а то, что
хозяйству не нужно, уходит в нижний бьеф. События расчёта выше говорят, сколько это:

In [ ]:
# @title Код: spill
#| label: spill
spills = system.nodes["reservoir"].events_of_type(WaterSpilled)  # the four-block run from the section above
spilled = sum(event.amount for event in spills) / YEARS  # Mm³ per year
spill_months = len({event.t for event in spills})
winter_share = sum(event.amount for event in spills if event.t % 12 < 6) / (spilled * YEARS)
print(f"Spilled: {spilled:.1f} Mm³ per year, in {spill_months} of {12 * YEARS} months; {winter_share:.0%} of it in winter")

In [ ]:
# @title Код: spill-text
#| label: spill-text
#| echo: false
river = sum(inflows) / YEARS
say(
    f"So {spilled / river:.0%} of the river never finds room in the reservoir: {winter_share:.0%} of it in winter, "
    f"when the farm needs almost nothing, the rest in wet summer months, when the release already covers the farm. "
    f"Either way the water runs through to the outlet unused. Two things "
    f"follow. In reality that water is not wasted: it is the river downstream, for nature and for the next user, "
    f"which this book simply does not count; the game *River flow for nature* is about exactly that. And the spill "
    f"is the reservoir's size at work: a larger reservoir would turn part of it into summer water, which is the "
    f"classic question of how big to build.",
    f"Demak, daryo suvining {spilled / river:.0%} i suv omboriga sigʻmaydi: uning {winter_share:.0%} i qishda, "
    f"xoʻjalikka deyarli suv kerak boʻlmaganda, qolgani esa sersuv yoz oylarida, chiqarilgan suv xoʻjalik ehtiyojini "
    f"allaqachon qoplaganda. Har ikki holda ham suv foydalanilmay quyi oqimga oqib oʻtadi. Bundan ikki "
    f"xulosa kelib chiqadi. Aslida bu suv behuda ketmaydi: bu quyi oqimdagi daryo, tabiat va keyingi isteʼmolchi uchun "
    f"suv — bu daftar uni shunchaki hisobga olmaydi; *Tabiat uchun daryo oqimi* oʻyini aynan shu haqda. Toshib ketish "
    f"esa suv ombori hajmining natijasi: kattaroq ombor uning bir qismini yozgi suvga aylantirgan boʻlardi — bu "
    f"qanday kattalikda qurish kerak degan klassik savol.",
    f"Итак, {spilled / river:.0%} стока реки не находит места в водохранилище: {winter_share:.0%} этого объёма "
    f"сбрасывается зимой, когда хозяйству почти ничего не нужно, остальное в многоводные летние месяцы, когда попуск "
    f"уже покрывает потребность хозяйства. В обоих случаях вода уходит в нижний бьеф без пользы. Из этого "
    f"следуют две вещи. В действительности эта вода не пропадает: это река ниже по течению, для природы и для "
    f"следующего водопользователя, просто этот учебник её не считает; игра *Вода для реки* как раз об этом. А сброс "
    f"— это следствие размера водохранилища: более крупное превратило бы часть его в летнюю воду; это классический "
    f"вопрос о том, какого размера строить.",
)

# Второй расчёт: лимитирование

Два плана лимитирования. Оба начинают лимитировать, когда в водохранилище меньше 16 млн м³. Один урезает подачу
немного (до 80 %), другой — сильно (до 50 %).

In [ ]:
# @title Код: fig-second-run
#| label: fig-second-run
#| fig-cap: "Three plans, same river. The grey line marks the trigger."
# A Plan holds the two knobs. Below 16 Mm³ in store, deliver 80 % (lightly) or 50 % (deep) of the summer need.
ration_lightly = simulate_plan(inflows, Plan(trigger=16, ration=0.8))
cut_deep = simulate_plan(inflows, Plan(trigger=16, ration=0.5))

# A dictionary of name -> run; the names become the panel titles
runs = {
    "Deliver in full until empty": no_rationing,
    "Ration lightly (below 16 Mm³, deliver 80 %)": ration_lightly,
    "Cut deep (below 16 Mm³, deliver 50 %)": cut_deep,
}
_ = plot_runs(runs, first_year=first_year)

Посмотрите на столбики слева. «Лёгкое лимитирование» превращает несколько высоких столбиков во множество низких
по 20 %, но самый высокий столбик в засуху остаётся. При «сильном урезании» ни один столбик не выше 50 %: худшее,
что случается, — само урезание, потому что водохранилище никогда не опустошается полностью. Цена видна в двух
других оценках.

In [ ]:
# @title Код: tbl-three-plans
#| label: tbl-three-plans
#| tbl-cap: "The three plans, scored."
#| echo: false
pd.DataFrame(
    [
        (name, run.scores.mean_shortage, run.scores.worst_month, run.scores.years_short)
        for name, run in runs.items()
    ],
    columns=["Plan", "Average shortage (Mm³/yr)", "Worst month (% short)", "Years short (of 100)"],
).style.hide(axis="index").format({"Average shortage (Mm³/yr)": "{:.1f}", "Worst month (% short)": "{:.0f}"})

In [ ]:
# @title Код: second-scores
#| label: second-scores
#| echo: false
light, deep = ration_lightly.scores, cut_deep.scores
deep_min = cut_deep.storage.min()  # the lowest the reservoir ever gets under the deep cut
say(
    f"- **How to read the storage panel.** The grey line is the trigger, 16 Mm³. Every summer month in which the "
    f"blue line is below it is a rationed month. Without rationing the line hits zero in the drought; under "
    f"\"Cut deep\" it never falls below {deep_min:.1f} Mm³, because halving the deliveries keeps water back.\n"
    f"- **How to read the bars.** Under \"Ration lightly\" most bars are 20 % tall: that is the ration itself, "
    f"in every year the trigger is reached. There are more of them ({light.years_short} years instead of "
    f"{s.years_short}), and the tallest one, {light.worst_month:.0f} %, is still there. Under \"Cut deep\" the bars "
    f"are 50 % at most and there are {deep.years_short} of them.\n"
    f"- **Rationing lightly does not help here.** The worst month is still {light.worst_month:.0f} % short: in the "
    f"worst drought the reservoir runs dry anyway. The farm is simply short more often "
    f"({light.years_short} years instead of {s.years_short}).\n"
    f"- **Cutting deep works, at a price.** The worst month improves from {s.worst_month:.0f} % to "
    f"{deep.worst_month:.0f} % short. But {deep.mean_shortage:.1f} Mm³ per year is not delivered instead of "
    f"{s.mean_shortage:.1f}, and the farm is short in {deep.years_short} years instead of {s.years_short}.\n\n"
    f"So there is no free lunch: a milder worst month costs water. How much, exactly?",
    f"- **Suv hajmi panelini qanday oʻqish.** Kulrang chiziq — chegara, 16 mln m³. Koʻk chiziq undan pastda boʻlgan "
    f"har bir yoz oyi — cheklangan oy. Cheklashsiz chiziq qurgʻoqchilikda nolga tushadi; \"Keskin kesish\" ostida "
    f"u hech qachon {deep_min:.1f} mln m³ dan pastga tushmaydi, chunki suv berishni yarmiga kamaytirish suvni saqlab "
    f"qoladi.\n"
    f"- **Ustunlarni qanday oʻqish.** \"Ozgina cheklash\" ostida koʻp ustunlar 20 % balandlikda: bu cheklashning "
    f"oʻzi, chegaraga yetilgan har bir yilda. Ular koʻproq ({s.years_short} oʻrniga {light.years_short} yil), eng "
    f"balandi esa, {light.worst_month:.0f} %, hali ham turibdi. \"Keskin kesish\" ostida ustunlar koʻpi bilan 50 % va "
    f"ular {deep.years_short} ta.\n"
    f"- **Ozgina cheklash bu yerda yordam bermaydi.** Eng yomon oy hali ham {light.worst_month:.0f} % taqchil: eng "
    f"qattiq qurgʻoqchilikda ombor baribir quriydi. Xoʻjalik shunchaki koʻproq taqchillik koʻradi "
    f"({s.years_short} oʻrniga {light.years_short} yil).\n"
    f"- **Keskin kesish ishlaydi, lekin bahosi bor.** Eng yomon oy {s.worst_month:.0f} % dan {deep.worst_month:.0f} % "
    f"taqchillikka yaxshilanadi. Lekin {s.mean_shortage:.1f} oʻrniga yiliga {deep.mean_shortage:.1f} mln m³ "
    f"yetkazilmaydi, va xoʻjalik {s.years_short} oʻrniga {deep.years_short} yilda taqchillik koʻradi.\n\n"
    f"Demak, tekin tushlik yoʻq: yumshoqroq eng yomon oy suvga tushadi. Aniq qanchaga?",
    f"- **Как читать панель с запасом.** Серая линия — порог, 16 млн м³. Каждый летний месяц, в который синяя "
    f"линия ниже порога, — месяц с лимитированием. Без лимитирования линия в засуху доходит до нуля; при "
    f"«сильном урезании» она никогда не опускается ниже {deep_min:.1f} млн м³, потому что половинная подача "
    f"сберегает воду.\n"
    f"- **Как читать столбики.** При «лёгком лимитировании» большинство столбиков высотой 20 %: это само "
    f"урезание, в каждый год, когда достигнут порог. Их больше ({light.years_short} лет вместо {s.years_short}), а "
    f"самый высокий, {light.worst_month:.0f} %, никуда не делся. При «сильном урезании» столбики не выше 50 %, и их "
    f"{deep.years_short}.\n"
    f"- **Лёгкое лимитирование здесь не помогает.** Худший месяц по-прежнему с дефицитом {light.worst_month:.0f} %: "
    f"в самую сильную засуху водохранилище всё равно пустеет. Хозяйство просто чаще испытывает дефицит "
    f"({light.years_short} лет вместо {s.years_short}).\n"
    f"- **Сильное урезание работает, но имеет цену.** Худший месяц улучшается с {s.worst_month:.0f} % до "
    f"{deep.worst_month:.0f} % дефицита. Но не подаётся {deep.mean_shortage:.1f} млн м³ в год вместо "
    f"{s.mean_shortage:.1f}, и хозяйство испытывает дефицит в {deep.years_short} годах вместо {s.years_short}.\n\n"
    f"Бесплатных обедов нет: более мягкий худший месяц стоит воды. Сколько именно?",
)

## Попробуйте свой план

In [ ]:
# @title Код: fig-my-plan
#| label: fig-my-plan
#| fig-cap: "Your plan against no rationing."
my_plan = Plan(trigger=22, ration=0.55)  # change these two numbers and run the cell again

mine = simulate_plan(inflows, my_plan)
_ = plot_runs({"Deliver in full until empty": no_rationing, "My plan": mine}, first_year=first_year)
print(mine.scores)  # the three scores of your plan

Порог может быть любым от 0 до 40 млн м³ (ёмкость), доля — любой от 0 до 1. Порог 40 означает «летом лимитировать
всегда».

На что смотреть: обе строки имеют одни и те же оси, так что сравнивайте их напрямую. Три оценки вашего плана
печатаются под рисунком. План лучше только тогда, когда он улучшает одну оценку, не ухудшая две другие; большинство
изменений меняют одно на другое. Две полезные крайности: порог 0 — «никогда не лимитировать» (снова первая
строка), порог 40 — «лимитировать в каждый летний месяц».

# Все планы сразу

> **Задача в формулах.** Месяцы нумеруются $t = 1, \dots, 12N$ за $N = 100$ лет; $m(t)$ — месяц внутри своего
> года, $S$ — шесть летних месяцев. В начале месяца $t$ в водохранилище $s_t$ воды, и в него приходит приток $q_t$;
> всё сверх ёмкости $K = 40$ млн м³ сбрасывается, затем вычитается потеря $\ell_t$ (нуль до раздела об
> испарении). Вода, которую видит правило, — $\hat s_t$. План — пара $\theta = (\tau, \rho)$, порог и доля;
> потребность хозяйства $d_m$ (0,83 млн м³ в зимний месяц, 7,5 в летний). Сброс и водный баланс:
>
$$
r_t(\theta) = \min\{\hat s_t,\; \rho_t\, d_{m(t)}\}, \qquad
\rho_t = \begin{cases} \rho & \text{if } m(t) \in S \text{ and } \hat s_t < \tau \\ 1 & \text{otherwise,} \end{cases}
$$
>
$$
\hat s_t = \min\{K,\; s_t + q_t\} - \ell_t, \qquad s_{t+1} = \hat s_t - r_t,
$$
>
> а дефицит месяца $t$ — $\delta_t = d_{m(t)} - r_t$. Два показателя, в млн м³ в год и в процентах:
>
$$
f_1(\theta) = \frac{1}{N}\sum_{t=1}^{12N} \delta_t(\theta), \qquad
f_2(\theta) = 100 \max_t \frac{\delta_t(\theta)}{d_{m(t)}},
$$
>
> и задача — сделать оба малыми одновременно:
>
$$
\min_{\theta \in [0,K] \times [0,1]} \big(f_1(\theta),\, f_2(\theta)\big),
$$
>
> Два числа не могут быть наименьшими одновременно, поэтому «решить» значит: найти планы, которые не превосходит
> никакой другой. План $\theta$ непревзойдён, если нет $\theta'$ с $f_1(\theta') \le f_1(\theta)$ и
> $f_2(\theta') \le f_2(\theta)$, причём хотя бы одно строго; множество непревзойдённых планов — фронт Парето.
> Перебор вычисляет $f$ в 21 × 21 значениях $\theta$; оптимизатор ищет по всему прямоугольнику.

Два рычага — достаточно мало, чтобы просто перебрать всё: 21 порог × 21 доля = 441 план, 441 расчёт в TaqSim.

In [ ]:
# @title Код: grid
#| label: grid
plans = plan_grid()  # 441 Plans: triggers 0, 2, 4, ... 40 Mm³ combined with rations 0, 0.05, 0.10, ... 1
table = score_plans(inflows, plans)  # runs each plan through TaqSim; one row per plan with its three scores

table.sample(5, random_state=1).sort_index().style.hide(axis="index").format(precision=2)  # five rows, as a taste

Каждая строка — один план: два рычага, три показателя и флаг `unbeaten`, объяснённый ниже. Все 441 расчёт
занимают несколько секунд.

Следующий рисунок показывает все 441 план на трёх картах, по одной на оценку. Каждая карта — шахматная доска:
порог по горизонтали, доля подачи по вертикали, одна клетка — один план, темнее — хуже. Левая карта: чем правее и
ниже (лимитировать раньше, подавать меньше), тем больше воды удерживается и так и не подаётся. Средняя карта: самые
светлые клетки, самые мягкие худшие месяцы, находятся справа вверху — высокий порог и подача около 60–75 %; тёмная
полоса внизу — «почти ничего не подавать, как только началось лимитирование». Правая карта: число дефицитных лет
растёт с порогом, потому что лимитирование включается в большем числе лет. Ни одна клетка не светлая на всех трёх
картах. В этом и есть компромисс.

In [ ]:
# @title Код: fig-maps
#| label: fig-maps
#| fig-cap: "Each score for every plan. Darker is worse. The three maps do not agree on where the good plans are."
#| echo: false
_ = plot_score_maps(table)

Теперь сопоставим два показателя. Каждый план становится одной точкой.

In [ ]:
# @title Код: fig-tradeoff
#| label: fig-tradeoff
#| fig-cap: "All 441 plans. Left is better (less water undelivered). Down is better (milder worst month). A plan is 'beaten' if another plan is at least as good on both counts and better on one."
# Every row of the table becomes a dot; `named` runs are circled and labelled
_ = plot_tradeoff(table, named={"deliver in full": no_rationing, "ration lightly": ration_lightly, "cut deep": cut_deep})

Чёрные точки — **непревзойдённые планы**. Для каждого из них единственный способ смягчить худший месяц — оставить
больше воды неподанной. Только эти планы стоит обсуждать; какой из них выбрать — вопрос суждения, а не расчёта.

In [ ]:
# @title Код: tradeoff-howto
#| label: tradeoff-howto
#| echo: false
front = front_of(table)
head, tail = front.iloc[0], front.iloc[-1]
say(
    f"How to read the chart: one dot per plan. Across, the average shortage, less is better; up, the worst month, "
    f"lower is better. The best corner is the bottom left, and no plan reaches it. The circled dots are the three "
    f"plans of the earlier sections: \"ration lightly\" sits almost on top of \"deliver in full\", \"cut deep\" is "
    f"on the staircase. The staircase joins the unbeaten plans. \"Plans lie further right\" means the axis is cut "
    f"off so that the interesting part stays readable.\n\n"
    f"The table below lists the staircase from its top-left end to its bottom-right end: from no rationing "
    f"({head['mean_shortage']:.2f} Mm³ per year undelivered, worst month {head['worst_month']:.0f} % short, "
    f"{head['years_short']:.0f} years short) to the deepest plan still worth having (trigger {tail['trigger']:.0f} Mm³, "
    f"ration {tail['ration']:.2f}: {tail['mean_shortage']:.2f} Mm³ per year, {tail['worst_month']:.0f} %, "
    f"{tail['years_short']:.0f} years short). Every step down the table buys a milder worst month with more water "
    f"left undelivered and more years with some shortage.",
    f"Diagrammani qanday oʻqish: har bir nuqta bitta reja. Koʻndalangiga — oʻrtacha taqchillik, kami yaxshi; yuqoriga "
    f"— eng yomon oy, pasti yaxshi. Eng yaxshi burchak — pastki chap, va hech bir reja unga yetmaydi. Doira ichidagi "
    f"nuqtalar — oldingi boʻlimlardagi uchta reja: \"ozgina cheklash\" deyarli \"toʻliq berish\" ustida turibdi, "
    f"\"keskin kesish\" zinapoyada. Zinapoya yengilmagan rejalarni birlashtiradi. \"Rejalar oʻngroqda\" degani — "
    f"qiziqarli qism oʻqilishi uchun oʻq kesilgan.\n\n"
    f"Quyidagi jadval zinapoyani yuqori chap uchidan pastki oʻng uchigacha sanab oʻtadi: cheklashsizdan (yiliga "
    f"{head['mean_shortage']:.2f} mln m³ yetkazilmagan, eng yomon oy {head['worst_month']:.0f} % taqchil, "
    f"{head['years_short']:.0f} taqchil yil) hali ham arziydigan eng chuqur rejagacha (chegara {tail['trigger']:.0f} "
    f"mln m³, cheklash {tail['ration']:.2f}: yiliga {tail['mean_shortage']:.2f} mln m³, {tail['worst_month']:.0f} %, "
    f"{tail['years_short']:.0f} taqchil yil). Jadvalda har bir qadam pastga yumshoqroq eng yomon oyni koʻproq "
    f"yetkazilmagan suv va koʻproq taqchil yil evaziga sotib oladi.",
    f"Как читать график: одна точка — один план. По горизонтали средний дефицит, меньше — лучше; по вертикали худший "
    f"месяц, ниже — лучше. Лучший угол — левый нижний, и ни один план до него не дотягивает. Обведённые точки — три "
    f"плана из предыдущих разделов: «лёгкое лимитирование» лежит почти поверх «подавать полностью», «сильное урезание» "
    f"— на лестнице. Лестница соединяет непревзойдённые планы. «Планы лежат правее» означает, что ось обрезана, чтобы "
    f"интересная часть оставалась читаемой.\n\n"
    f"Таблица ниже перечисляет лестницу от её левого верхнего конца до правого нижнего: от отсутствия лимитирования "
    f"({head['mean_shortage']:.2f} млн м³ в год не подано, худший месяц {head['worst_month']:.0f} % дефицита, "
    f"{head['years_short']:.0f} дефицитных лет) до самого глубокого плана, который ещё имеет смысл (порог "
    f"{tail['trigger']:.0f} млн м³, подача {tail['ration']:.2f}: {tail['mean_shortage']:.2f} млн м³ в год, "
    f"{tail['worst_month']:.0f} %, {tail['years_short']:.0f} дефицитных лет). Каждый шаг вниз по таблице покупает "
    f"более мягкий худший месяц ценой большего объёма неподанной воды и большего числа лет с дефицитом.",
)

In [ ]:
# @title Код: tbl-front
#| label: tbl-front
#| tbl-cap: "The unbeaten plans, from least water undelivered to mildest worst month."
#| echo: false
front = front_of(table)
front[["trigger", "ration", "mean_shortage", "worst_month", "years_short"]].rename(
    columns={
        "trigger": "Trigger (Mm³)",
        "ration": "Ration",
        "mean_shortage": "Average shortage (Mm³/yr)",
        "worst_month": "Worst month (% short)",
        "years_short": "Years short (of 100)",
    }
).style.hide(axis="index").format(
    {"Trigger (Mm³)": "{:.0f}", "Ration": "{:.2f}", "Average shortage (Mm³/yr)": "{:.2f}", "Worst month (% short)": "{:.0f}"}
)

# Поручаем поиск оптимизатору

Перебор сработал, потому что рычагов всего два. У настоящего правила управления водохранилищем их десятки (свои
порог и доля для каждого месяца, несколько водохранилищ, несколько водопользователей). При десяти рычагах по 21
значению «всё» — это больше расчётов, чем кто-либо готов ждать.

Вместо перебора ищет оптимизатор (TaqSim использует метод NSGA-II). Он начинает со случайных планов, оставляет
непревзойдённые, создаёт новые планы, смешивая и слегка изменяя их, и повторяет. Мы даём ему систему и две цели:

In [ ]:
# @title Код: objectives-source
#| label: objectives-source
#| echo: false
show_source(objectives)

У каждой `Objective` есть имя, направление и функция, превращающая рассчитанную систему в одно число.

In [ ]:
# @title Код: optimise
#| label: optimise
found = optimise_plans(
    inflows,
    pop_size=40,  # the optimizer works with 40 plans at a time
    generations=40,  # and improves them 40 times: 1 600 TaqSim runs in all
    seed=42,  # the starting plans are random; a fixed seed makes the result reproducible
)
print(f"The optimizer returned {len(found)} plans, {len(front_of(found))} of them different.")

Внутри `optimise_plans` — один вызов TaqSim, `optimize(system, objectives, timesteps, pop_size, generations,
seed)`, после чего каждый возвращённый план пересчитывается, чтобы снять его показатели. Таблица результата имеет
те же столбцы, что и таблица перебора выше.

In [ ]:
# @title Код: fig-optimizer
#| label: fig-optimizer
#| fig-cap: "The optimizer's plans (blue) on top of the 441 grid plans. It finds the same curve and fills the gaps, because it is not tied to the grid's steps."
_ = plot_tradeoff(table, found=found)

In [ ]:
# @title Код: optimizer-howto
#| label: optimizer-howto
#| echo: false
distinct = len(front_of(found))
grid_best = front["worst_month"].min()
say(
    f"How to read it: the blue dots are the {len(found)} plans the optimizer finished with, {distinct} of them "
    f"different. Most sit on or just beside the black staircase, and several lie between its steps: the optimizer may "
    f"choose any trigger and any ration, not only the grid's steps of 2 Mm³ and 5 %. A few blue dots reach further "
    f"right than the last black one: the optimizer also found plans with a worst month below {grid_best:.0f} %, at a "
    f"high price in water. It did this with 1 600 runs and without ever seeing the grid.",
    f"Buni qanday oʻqish: koʻk nuqtalar — optimallashtiruvchi yakunlagan {len(found)} reja, ulardan {distinct} tasi "
    f"har xil. Koʻpchiligi qora zinapoyaning ustida yoki yonida, bir nechtasi esa uning pogʻonalari orasida: "
    f"optimallashtiruvchi istalgan chegara va istalgan cheklashni tanlashi mumkin, toʻrning 2 mln m³ va 5 % lik "
    f"qadamlarini emas. Bir nechta koʻk nuqta oxirgi qora nuqtadan oʻngroqqa yetadi: optimallashtiruvchi eng yomon "
    f"oyi {grid_best:.0f} % dan past rejalarni ham topdi, suv boʻyicha qimmat narxda. U buni 1 600 ta hisob bilan va "
    f"toʻrni umuman koʻrmasdan qildi.",
    f"Как это читать: синие точки — {len(found)} планов, с которыми оптимизатор закончил, {distinct} из них разные. "
    f"Большинство лежат на чёрной лестнице или рядом с ней, а несколько — между её ступенями: оптимизатор может "
    f"выбрать любой порог и любую долю подачи, а не только шаги сетки в 2 млн м³ и 5 %. Несколько синих точек "
    f"уходят правее последней чёрной: оптимизатор нашёл и планы с худшим месяцем ниже {grid_best:.0f} %, по высокой "
    f"цене в воде. Он сделал это за 1 600 прогонов, ни разу не увидев сетку.",
)

# На что обратить внимание

In [ ]:
# @title Код: notice
#| label: notice
#| echo: false
first, last = front.iloc[0], front.iloc[-1]
knee = front.loc[(front["worst_month"] - 50.0).abs().idxmin()]
beaten = int((~table["unbeaten"]).sum())
best_volume = table.loc[table["mean_shortage"].idxmin()]
say(
    f"**1. What you count decides the answer.** If the only goal is to deliver as much water as possible, there is "
    f"one best plan and it is *no rationing* (average shortage {best_volume['mean_shortage']:.2f} Mm³ per year, the "
    f"lowest of all 441 plans). The trade-off only appears once you also care about the worst month. Choosing the "
    f"goals is the most important modelling decision, and it is not a technical one.\n\n"
    f"**2. The curve bends.** Going from {first['worst_month']:.0f} % to {knee['worst_month']:.0f} % short in the "
    f"worst month costs {knee['mean_shortage'] - first['mean_shortage']:.1f} Mm³ per year. Going on from "
    f"{knee['worst_month']:.0f} % to {last['worst_month']:.0f} % costs another "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} Mm³ per year. The first steps are cheap, the last ones "
    f"very expensive. A plan near the bend is the natural place to start the discussion.\n\n"
    f"**3. Being short more often is part of the price.** Along the curve the number of short years rises from "
    f"{first['years_short']:.0f} to {last['years_short']:.0f} of 100.\n\n"
    f"**4. Most plans are simply bad.** {beaten} of the 441 plans are beaten, including the reasonable-sounding "
    f"\"ration lightly\". Intuition is a poor guide here; the search is not.",
    f"**1. Nimani hisoblashingiz javobni belgilaydi.** Agar yagona maqsad iloji boricha koʻp suv yetkazish boʻlsa, "
    f"bitta eng yaxshi reja bor va bu — *cheklash yoʻq* (oʻrtacha taqchillik yiliga "
    f"{best_volume['mean_shortage']:.2f} mln m³, barcha 441 reja ichida eng kami). Murosa faqat eng yomon oy ham "
    f"sizga muhim boʻlgandagina paydo boʻladi. Maqsadlarni tanlash — modellashtirishdagi eng muhim qaror, va u "
    f"texnik qaror emas.\n\n"
    f"**2. Egri chiziq bukiladi.** Eng yomon oyda {first['worst_month']:.0f} % dan {knee['worst_month']:.0f} % "
    f"taqchillikka oʻtish yiliga {knee['mean_shortage'] - first['mean_shortage']:.1f} mln m³ ga tushadi. "
    f"{knee['worst_month']:.0f} % dan {last['worst_month']:.0f} % gacha davom etish yana yiliga "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} mln m³ ga tushadi. Birinchi qadamlar arzon, oxirgilari "
    f"juda qimmat. Muhokamani bukilish yaqinidagi rejadan boshlash tabiiy.\n\n"
    f"**3. Koʻproq taqchillik koʻrish — bahoning bir qismi.** Egri chiziq boʻylab taqchil yillar soni 100 dan "
    f"{first['years_short']:.0f} tadan {last['years_short']:.0f} tagacha oshadi.\n\n"
    f"**4. Koʻpchilik rejalar shunchaki yomon.** 441 rejadan {beaten} tasi ustun kelingan, shu jumladan oqilona "
    f"tuyulgan \"ozgina cheklash\" ham. Bu yerda sezgi yomon yoʻlboshchi; qidiruv esa yaxshi.",
    f"**1. Ответ определяет то, что вы считаете.** Если единственная цель — подать как можно больше воды, есть один "
    f"лучший план, и это *без лимитирования* (средний дефицит {best_volume['mean_shortage']:.2f} млн м³ в год, "
    f"наименьший из всех 441 планов). Компромисс появляется только тогда, когда вам важен и худший месяц. Выбор "
    f"целей — самое важное решение при моделировании, и оно не техническое.\n\n"
    f"**2. Кривая изгибается.** Переход от {first['worst_month']:.0f} % к {knee['worst_month']:.0f} % дефицита в "
    f"худшем месяце стоит {knee['mean_shortage'] - first['mean_shortage']:.1f} млн м³ в год. Дальнейший переход от "
    f"{knee['worst_month']:.0f} % к {last['worst_month']:.0f} % стоит ещё "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} млн м³ в год. Первые шаги дёшевы, последние очень дороги. "
    f"План возле изгиба — естественная отправная точка для обсуждения.\n\n"
    f"**3. Более частый дефицит — часть цены.** Вдоль кривой число дефицитных лет растёт с "
    f"{first['years_short']:.0f} до {last['years_short']:.0f} из 100.\n\n"
    f"**4. Большинство планов просто плохи.** {beaten} из 441 планов превзойдены, включая разумное на слух «лёгкое "
    f"лимитирование». Интуиция здесь плохой советчик; поиск — хороший.",
)

# Ещё один блок: испарение

До сих пор вода, сбережённая в водохранилище, ждала там бесплатно. В жарком климате так не бывает: часть её
испаряется. В TaqSim это ещё один небольшой класс, *правило потерь*, подключаемый к водохранилищу:

In [ ]:
# @title Код: evaporation-source
#| label: evaporation-source
#| echo: false
show_source(SummerEvaporation)

`calculate(...)` TaqSim вызывает раз в месяц, после того как приток сохранён, и перед сбросом. Метод возвращает,
что потеряно и почему; TaqSim вычитает это из объёма и записывает событие `WaterLost`.

In [ ]:
# @title Код: fig-evaporation
#| label: fig-evaporation
#| fig-cap: "The unbeaten plans without and with evaporation. With evaporation every plan delivers less, and a mild worst month becomes much more expensive."
# Same 441 plans, same river, but now the reservoir loses 3 % of its content in each summer month
with_evaporation = score_plans(inflows, plan_grid(), evaporation_share=EVAPORATION_SHARE_EXAMPLE)

# Draw the unbeaten plans of both tables on one chart
_ = plot_front_shift({"no evaporation": table, "3 % of the stored water evaporates each summer month": with_evaporation})

In [ ]:
# @title Код: evaporation-text
#| label: evaporation-text
#| echo: false
front_e = front_of(with_evaporation)
at_50 = front[front["worst_month"] <= 50.0].iloc[0]
at_50_e = front_e[front_e["worst_month"] <= 50.0].iloc[0]
say(
    f"The two staircases are the unbeaten plans without (blue) and with (orange) evaporation. The orange one lies to "
    f"the right of and above the blue one: every plan delivers less, and the same worst month costs more water. It "
    f"also stops higher: with evaporation no plan gets the worst month below {front_e['worst_month'].min():.0f} %, "
    f"against {front['worst_month'].min():.0f} % without. "
    f"The 3 % is an illustrative figure, not a measurement. The effect is large: holding the worst month to 50 % "
    f"short costs {at_50['mean_shortage']:.1f} Mm³ per year without evaporation and "
    f"{at_50_e['mean_shortage']:.1f} Mm³ per year with it. Saving water for later is worth less when the saved "
    f"water does not keep.",
    f"Ikkita zinapoya — bugʻlanishsiz (koʻk) va bugʻlanish bilan (toʻq sariq) yengilmagan rejalar. Toʻq sariq "
    f"zinapoya koʻkning oʻngida va yuqorisida: har bir reja kamroq yetkazadi, va bir xil eng yomon oy koʻproq suvga "
    f"tushadi. U yuqoriroqda ham toʻxtaydi: bugʻlanish bilan hech bir reja eng yomon oyni "
    f"{front_e['worst_month'].min():.0f} % dan pastga tushira olmaydi, bugʻlanishsiz esa "
    f"{front['worst_month'].min():.0f} % gacha. "
    f"3 % — oʻlchov emas, tushuntirish uchun olingan raqam. Taʼsiri katta: eng yomon oyni 50 % taqchillikda "
    f"ushlab turish bugʻlanishsiz yiliga {at_50['mean_shortage']:.1f} mln m³, bugʻlanish bilan esa yiliga "
    f"{at_50_e['mean_shortage']:.1f} mln m³ ga tushadi. Saqlangan suv saqlanmasa, uni keyinga asrash kamroq "
    f"qadrli.",
    f"Две лестницы — непревзойдённые планы без испарения (синяя) и с испарением (оранжевая). Оранжевая лежит правее "
    f"и выше синей: каждый план подаёт меньше, а тот же худший месяц стоит больше воды. Она и обрывается выше: с "
    f"испарением ни один план не опускает худший месяц ниже {front_e['worst_month'].min():.0f} %, без испарения — "
    f"до {front['worst_month'].min():.0f} %. "
    f"3 % — иллюстративное число, не измерение. Эффект велик: удержать худший месяц на уровне 50 % дефицита стоит "
    f"{at_50['mean_shortage']:.1f} млн м³ в год без испарения и {at_50_e['mean_shortage']:.1f} млн м³ в год с "
    f"ним. Беречь воду на потом менее выгодно, когда сбережённая вода не сохраняется.",
)

# Ещё один водопользователь: зимние поля

Водохранилище наполняется зимой, а зима в Зарафшанском оазисе не пустует. Поливают озимую пшеницу, а перед
весенним севом промывают засолённые поля: их затапливают, чтобы соль ушла ниже корней. И то и другое берёт воду из
того же водохранилища в те месяцы, когда оно должно наполняться. Добавим этого водопользователя пятым блоком, на
том же канале ниже хозяйства: **зимние поля**. Они просят 2,5 млн м³ в каждый зимний месяц, 15 млн м³ за зиму, и
ничего летом. Число иллюстративное, как и доля испарения.

In [ ]:
# @title Код: fig-winter-network
#| label: fig-winter-network
#| fig-cap: "Five blocks. The winter fields sit on the canal below the farm: in winter they take what the reservoir releases for them, and whatever the reservoir spills."
_ = plot_network(winter=True)

Каждую зиму перед оператором встаёт новый вопрос. Вода, отданная зимним полям, уходит раньше, чем кто-либо
узнает, каким будет летнее таяние. Отдать полностью — и слабое таяние оставит хозяйство без воды в июле.
Придержать — и в большинстве зим окажется, что вода была не нужна. Правило получает ещё два рычага, для зимы:

- **Зимний порог** (млн м³): когда *вода на горизонте* ниже него, зимние поля лимитируются. Пока вода на
  горизонте — это просто запас в водохранилище.
- **Зимняя доля** (от 0 до 1): какую часть своей потребности зимние поля тогда получают.

Летние рычаги остаются прежними. До конца этой тетради мы фиксируем их на «сильном урезании» (порог 16 млн м³,
доля 0,5), плане из второго расчёта, чтобы менялось только зимнее решение.

In [ ]:
# @title Код: winter-rule-source
#| label: winter-rule-source
#| echo: false
show_source(WinterRelease)

In [ ]:
# @title Код: winter-two-runs
#| label: winter-two-runs
# Summer "cut deep" in both; the winter fields served in full, or given nothing when the store is below 24 Mm³
winter_in_full = simulate_winter(inflows, WINTER_IN_FULL)
hold_back = simulate_winter(inflows, WinterPlan(trigger=24.0, ration=0.0))
for name, run in {"Winter fields served in full": winter_in_full, "Hold back below 24 Mm³": hold_back}.items():
    print(
        f"{name:30s} farm short {run.scores.summer_shortage:.1f} Mm³/yr, "
        f"winter fields short {run.scores.winter_shortage:.1f} Mm³/yr"
    )

In [ ]:
# @title Код: winter-two-runs-text
#| label: winter-two-runs-text
#| echo: false
full, back = winter_in_full.scores, hold_back.scores
say(
    f"Served in full, the winter fields get almost all of their 15 Mm³, and the farm's shortage rises from "
    f"{cut_deep.scores.mean_shortage:.1f} Mm³ a year (the same summer plan without the winter fields) to "
    f"{full.summer_shortage:.1f}. Holding back below 24 Mm³ gives the farm back most of that, "
    f"{full.summer_shortage - back.summer_shortage:.1f} Mm³ a year, at a price of "
    f"{back.winter_shortage - full.winter_shortage:.1f} Mm³ a year for the winter fields. Even \"nothing\" is not "
    f"quite nothing: in wet winters the reservoir spills, and the spilled water reaches the fields anyway.\n\n"
    f"The two goals are now the two users: the water the farm did not get in summer, and the water the winter "
    f"fields did not get in winter, both in Mm³ a year, both less is better. As before, we try every winter plan: "
    f"triggers from 0 to 40 Mm³ in steps of 1, rations in steps of 0.05, 861 plans.",
    f"Toʻliq berilganda qishki dalalar oʻzining 15 mln m³ ining deyarli hammasini oladi, xoʻjalikning taqchilligi "
    f"esa yiliga {cut_deep.scores.mean_shortage:.1f} mln m³ dan (qishki dalalarsiz oʻsha yozgi reja) "
    f"{full.summer_shortage:.1f} ga koʻtariladi. 24 mln m³ dan pastda ushlab turish xoʻjalikka buning koʻp qismini, "
    f"yiliga {full.summer_shortage - back.summer_shortage:.1f} mln m³ ni qaytaradi, qishki dalalar uchun esa yiliga "
    f"{back.winter_shortage - full.winter_shortage:.1f} mln m³ ga tushadi. Hatto \"hech narsa\" ham toʻliq hech "
    f"narsa emas: sersuv qishlarda ombor toshadi va toshgan suv baribir dalalarga yetib boradi.\n\n"
    f"Endi ikki maqsad — ikki isteʼmolchi: xoʻjalik yozda olmagan suv va qishki dalalar qishda olmagan suv, "
    f"ikkalasi ham yiliga mln m³ da, ikkalasida ham kam boʻlgani yaxshi. Avvalgidek, har bir qishki rejani sinab "
    f"koʻramiz: chegaralar 0 dan 40 mln m³ gacha 1 qadam bilan, ulushlar 0,05 qadam bilan, 861 ta reja.",
    f"При полной подаче зимние поля получают почти все свои 15 млн м³, а дефицит хозяйства растёт с "
    f"{cut_deep.scores.mean_shortage:.1f} млн м³ в год (тот же летний план без зимних полей) до "
    f"{full.summer_shortage:.1f}. Придерживание ниже 24 млн м³ возвращает хозяйству большую часть этого, "
    f"{full.summer_shortage - back.summer_shortage:.1f} млн м³ в год, ценой "
    f"{back.winter_shortage - full.winter_shortage:.1f} млн м³ в год для зимних полей. Даже «ничего» — не совсем "
    f"ничего: в многоводные зимы водохранилище сбрасывает излишек, и эта вода всё равно доходит до полей.\n\n"
    f"Две цели — теперь два водопользователя: вода, которую хозяйство не получило летом, и вода, которую зимние "
    f"поля не получили зимой, обе в млн м³ в год, обе — чем меньше, тем лучше. Как и раньше, перебираем все "
    f"зимние планы: пороги от 0 до 40 млн м³ с шагом 1, доли с шагом 0,05, 861 план.",
)

> **Зимняя задача в формулах.** Летний план зафиксирован: $\theta^* = (16;\ 0{,}5)$; зимний план —
> $\psi = (\tau_w, \rho_w)$. Зимние поля просят $w_m = 2{,}5$ млн м³ в зимний месяц и ничего летом. В зимний
> месяц правило сравнивает воду на горизонте $v_t$ с зимним порогом: пока только запас, в следующем разделе —
> запас плюс прогноз $F_y$ предстоящего лета. Сначала обслуживается хозяйство, остаток получают зимние поля:
>
$$
r_t = \min\{\hat s_t,\; d_{m(t)} + \rho^{w}_t\, w_{m(t)}\}, \qquad
\rho^{w}_t = \begin{cases} \rho_w & \text{if } v_t < \tau_w \\ 1 & \text{otherwise,} \end{cases} \qquad
v_t = \hat s_t \;\text{ or }\; \hat s_t + F_{y(t)},
$$
>
> Летние месяцы подчиняются летнему правилу выше. Два показателя — дефициты двух водопользователей, в млн м³ в
> год; задача того же рода, что и раньше, по $\tau_w \in [0, 40]$ (с прогнозом $[0, 120]$) и $\rho_w \in [0, 1]$:
>
$$
g_1(\psi) = \frac{1}{N}\sum_{t:\, m(t) \in S} \delta^{\text{farm}}_t, \qquad
g_2(\psi) = \frac{1}{N}\sum_{t:\, m(t) \notin S} \delta^{\text{fields}}_t, \qquad
\min_{\psi} \big(g_1(\psi),\, g_2(\psi)\big).
$$

In [ ]:
# @title Код: fig-winter-front
#| label: fig-winter-front
#| fig-cap: "The unbeaten winter plans when the winter decision looks at the storage alone. Left is better for the farm, down is better for the winter fields."
# 861 winter plans, summer fixed at "cut deep"; the water in sight is the storage, so triggers run to 40 Mm³
storage_only = score_winter_plans(inflows, winter_grid(n_trigger=41, trigger_max=CAPACITY_MM3))
_ = plot_winter_fronts({"winter decided on the storage alone": storage_only})

In [ ]:
# @title Код: winter-front-text
#| label: winter-front-text
#| echo: false
at_2 = least_winter_shortage(storage_only, 2.0)
at_3 = least_winter_shortage(storage_only, 3.0)
say(
    f"The same bent curve as in summer. Its bottom-right end is \"serve in full\": the winter fields lack almost "
    f"nothing, the farm is short {full.summer_shortage:.1f} Mm³ a year. Moving up and left, each step gives the "
    f"farm back some water and takes more from the winter fields. To hold the farm's shortage to 3 Mm³ a year, the "
    f"winter fields must give up {at_3:.1f} Mm³ a year; to hold it to 2, {at_2:.1f}. The operator who only "
    f"watches the storage pays this price in every winter the reservoir is low, including the many that a strong "
    f"melt would have rescued.",
    f"Yozdagi kabi bukilgan egri chiziq. Uning pastki oʻng uchi — \"toʻliq berish\": qishki dalalarga deyarli hech "
    f"narsa yetishmaydi, xoʻjalik esa yiliga {full.summer_shortage:.1f} mln m³ taqchil. Yuqoriga va chapga "
    f"yurganda har bir qadam xoʻjalikka bir oz suv qaytaradi va qishki dalalardan koʻproq oladi. Xoʻjalik "
    f"taqchilligini yiliga 3 mln m³ da ushlab turish uchun qishki dalalar yiliga {at_3:.1f} mln m³ dan voz kechishi "
    f"kerak; 2 da ushlab turish uchun — {at_2:.1f}. Faqat ombordagi suvga qaraydigan operator bu narxni ombor kam "
    f"suvli boʻlgan har bir qishda toʻlaydi, shu jumladan kuchli erish qutqarib qoladigan koʻplab qishlarda ham.",
    f"Та же изогнутая кривая, что и летом. Её нижний правый конец — «подавать полностью»: зимним полям почти "
    f"ничего не не хватает, хозяйству не хватает {full.summer_shortage:.1f} млн м³ в год. Двигаясь вверх и влево, "
    f"каждый шаг возвращает хозяйству немного воды и отнимает больше у зимних полей. Чтобы удержать дефицит "
    f"хозяйства на 3 млн м³ в год, зимние поля должны отдать {at_3:.1f} млн м³ в год; чтобы удержать на 2 — "
    f"{at_2:.1f}. Оператор, который смотрит только на запас, платит эту цену каждую зиму, когда водохранилище "
    f"маловодно, включая те многие зимы, которые сильное таяние всё равно бы спасло.",
)

# Прогноз лета, решение зимой

Оператор придерживает воду, потому что лето неизвестно. Но оно неизвестно не полностью. К концу зимы большая часть
снега, который растает летом, уже выпала, и гидрометеорологические службы Центральной Азии каждую весну
прогнозируют сток своих рек за апрель–сентябрь по снежному покрову, зимним осадкам и стоку прошедших месяцев
(Apel et al., 2018). Предположим, что у нашего оператора в начале каждой зимы есть такой прогноз: одно число,
приток предстоящего лета. В действительности он приходит в конце зимы, когда снег уже выпал; здесь оператор
получает его в начале зимы — это лучший случай, а упражнение 6 спрашивает, чего стоит более поздний прогноз.

Правилу нужно одно изменение. Вода на горизонте — это теперь не только запас, а запас **плюс прогноз**, и с ним
сравнивается зимний порог. Больше ничего не меняется: те же рычаги, тот же летний план, та же река. Поскольку в
воду на горизонте теперь входит целое лето, порог доходит до 120 млн м³ вместо 40.

Начнём с лучшего случая — прогноза, который всегда верен. Он показывает, сколько прогноз вообще может здесь
стоить.

In [ ]:
# @title Код: fig-forecast-front
#| label: fig-forecast-front
#| fig-cap: "The unbeaten winter plans when the winter decision looks at the storage alone (orange) and at the storage plus a perfect forecast of the coming summer (blue)."
perfect = seasonal_forecast(inflows, skill=1.0)  # one number per year: the coming summer's inflow, exactly
with_perfect = score_winter_plans(inflows, winter_grid(n_trigger=41), forecast=perfect)  # 861 plans, triggers to 120
_ = plot_winter_fronts({"storage plus a perfect forecast": with_perfect, "storage alone": storage_only})

In [ ]:
# @title Код: fig-winter-decisions
#| label: fig-winter-decisions
#| fig-cap: "Two winter plans with the same summer shortage, 2 Mm³ a year at most, in the twenty years around the worst drought. Top: what the winter fields received each winter. Bottom: the summer that followed."
# From each curve, the plan that holds the farm's shortage to 2 Mm³ a year with the least loss for the winter fields
chosen = {
    "storage plus a perfect forecast": winter_plan_within(with_perfect, 2.0),
    "storage alone": winter_plan_within(storage_only, 2.0),
}
runs = {
    label: simulate_winter(inflows, winter, forecast=perfect if "forecast" in label else None)
    for label, winter in chosen.items()
}
_ = plot_winter_decisions(runs, inflows, first_year=first_year)

In [ ]:
# @title Код: forecast-front-text
#| label: forecast-front-text
#| echo: false
at_2_perfect = least_winter_shortage(with_perfect, 2.0)
at_3_perfect = least_winter_shortage(with_perfect, 3.0)
gain_2 = (at_2 - at_2_perfect) / at_2 * 100.0
say(
    f"- **The blue curve lies below the orange one.** At every summer shortage the farm accepts, the winter fields "
    f"lose less. Holding the farm to 2 Mm³ a year costs the winter fields {at_2:.1f} Mm³ a year without the "
    f"forecast and {at_2_perfect:.1f} with it, {gain_2:.0f} % less; at 3 Mm³ a year, {at_3:.1f} against "
    f"{at_3_perfect:.1f}.\n"
    f"- **Why.** Look at the two plans in the second figure. The plan without the forecast holds back whenever the "
    f"reservoir is low, which is mostly the winter *after* a dry summer. The plan with the forecast holds back in "
    f"the winters *before* a weak summer and serves in full before a strong one. It rations fewer winters, and the "
    f"right ones.\n"
    f"- **This gap is the ceiling.** No real forecast can do better than one that is always right. Whatever a "
    f"forecast service can offer is worth at most this much here: {at_2 - at_2_perfect:.1f} Mm³ a year for the "
    f"winter fields at a summer shortage of 2. In a system where more decisions are taken before the melt, the "
    f"ceiling is higher.",
    f"- **Koʻk egri chiziq toʻq sarigʻidan pastda yotadi.** Xoʻjalik rozi boʻlgan har bir yozgi taqchillikda qishki "
    f"dalalar kamroq yoʻqotadi. Xoʻjalikni yiliga 2 mln m³ da ushlab turish qishki dalalarga prognozsiz yiliga "
    f"{at_2:.1f} mln m³, prognoz bilan {at_2_perfect:.1f} ga tushadi, yaʼni {gain_2:.0f} % kam; yiliga 3 mln m³ "
    f"da — {at_3:.1f} ga qarshi {at_3_perfect:.1f}.\n"
    f"- **Nima uchun.** Ikkinchi rasmdagi ikki rejaga qarang. Prognozsiz reja ombor kam suvli boʻlganda ushlab "
    f"turadi, bu esa asosan quruq yozdan *keyingi* qish. Prognozli reja kuchsiz yozdan *oldingi* qishlarda ushlab "
    f"turadi va kuchli yozdan oldin toʻliq beradi. U kamroq qishni cheklaydi, va aynan keraklilarini.\n"
    f"- **Bu farq — shift.** Hech bir haqiqiy prognoz hamisha toʻgʻri chiqadiganidan yaxshiroq boʻla olmaydi. "
    f"Prognoz xizmati taklif qila oladigan narsa bu yerda eng koʻpi bilan shuncha turadi: yozgi taqchillik 2 "
    f"boʻlganda qishki dalalar uchun yiliga {at_2 - at_2_perfect:.1f} mln m³. Erishdan oldin koʻproq qaror "
    f"qabul qilinadigan tizimda shift balandroq.",
    f"- **Синяя кривая лежит ниже оранжевой.** При любом летнем дефиците, на который соглашается хозяйство, зимние "
    f"поля теряют меньше. Удержать хозяйство на 2 млн м³ в год стоит зимним полям {at_2:.1f} млн м³ в год без "
    f"прогноза и {at_2_perfect:.1f} с ним, на {gain_2:.0f} % меньше; при 3 млн м³ в год — {at_3:.1f} против "
    f"{at_3_perfect:.1f}.\n"
    f"- **Почему.** Посмотрите на два плана на второй рисунке. План без прогноза придерживает воду всякий раз, "
    f"когда водохранилище маловодно, а это в основном зима *после* сухого лета. План с прогнозом придерживает "
    f"воду в зимы *перед* слабым летом и подаёт полностью перед сильным. Он лимитирует меньше зим, и именно те, "
    f"что нужно.\n"
    f"- **Этот разрыв — потолок.** Ни один реальный прогноз не может быть лучше того, что всегда верен. Что бы ни "
    f"предложила прогностическая служба, здесь это стоит самое большее столько: {at_2 - at_2_perfect:.1f} млн м³ в "
    f"год для зимних полей при летнем дефиците 2. В системе, где больше решений принимается до таяния, потолок "
    f"выше.",
)

# Прогноз, который может ошибаться

**Идея.** В прошлом разделе оператор получил прогноз, который всегда верен. Ни один настоящий прогноз таким не
бывает. Вопрос этого раздела — тот, с которым на деле сталкивается водное ведомство: *прогноз, который иногда
ошибается, — стоит ли на него опираться, и насколько?* Житейский вариант знаком каждому. Прогноз погоды,
который сбывается семь раз из десяти, далёк от идеала, и всё же вы берёте зонт, когда он обещает дождь, и
правильно делаете: вы действуете по нему иначе, чем по идеальному, с меньшим доверием, но не выбрасываете его.
Мы хотим увидеть то же самое для водохранилища, в числах.

**Как мы это строим.** Купить сто лет настоящих прогнозов для синтетической реки нельзя, поэтому мы изготавливаем
прогнозы известного качества. Берём лето, которое действительно наступит, размываем его случайным шумом и даём
разумному прогнозисту сделать лучшую оценку, на какую он способен по размытой картине. Один рычаг задаёт степень
размытия: мы называем его **качеством** прогноза, от 0 до 1.

- Качество 1: размытия нет. Прогнозист видит лето в точности; это идеальный прогноз прошлого раздела.
- Качество 0: сплошное размытие. Прогнозист не видит ничего полезного и не может сделать ничего лучше, чем каждый
  год говорить «среднее лето».
- Между ними: прогнозист что-то видит, и прогнозы отчасти следуют за настоящими лётами. Техническое название
  качества — коэффициент корреляции между прогнозами и исходами.

`seasonal_forecast` строит такие прогнозы. Важно одно свойство: прогнозы честны. Прогнозист, который знает мало,
говорит что-то близкое к среднему; тот, кто знает много, решается сказать «очень сухое лето». В среднем оценка не
слишком смела и не слишком робка — именно так ведёт себя статистический прогноз, подобранный по прошлым годам.
Правило оператора — то, что вы знаете: оно прибавляет прогноз к воде на горизонте и сравнивает сумму с зимним
порогом. Два примера того, как выглядят прогнозы:

> **Прогноз в формулах.** Летний приток года $y$ — $Q_y = e^{\mu + \sigma z_y}$, где $z_y$ — стандартизованная
> величина (река логнормальна). Прогнозист видит предстоящее лето сквозь шум $\varepsilon_y$ и выдаёт лучшую оценку
> по тому, что увидел:
>
$$
\tilde z_y = \kappa\, z_y + \sqrt{1-\kappa^2}\; \varepsilon_y, \qquad
F_y = \mathbb{E}\big[Q_y \mid \tilde z_y\big] = \exp\!\Big(\mu + \sigma \kappa \tilde z_y + \tfrac{1}{2}\sigma^2 (1-\kappa^2)\Big),
$$
>
> так что корреляция между увиденным и истиной — это качество $\kappa$, прогноз при $\kappa = 0$ — многолетнее
> среднее, при $\kappa = 1$ — само лето. Ценность прогноза измеряется при фиксированном летнем дефиците $\bar c$
> (ниже 2 млн м³ в год): зимняя вода, которую он сберегает, ничего не стоя хозяйству,
>
$$
V(\kappa) = g_2^{*}(\text{no forecast}) - g_2^{*}(\kappa), \qquad
g_2^{*}(\cdot) = \min\big\{\, g_2(\psi) : g_1(\psi) \le \bar c \,\big\}.
$$
>
> $V(1)$, ценность идеального прогноза, — потолок.

In [ ]:
# @title Код: fig-forecast-skill
#| label: fig-forecast-skill
#| fig-cap: "One hundred summers and their forecasts, for a forecast of skill 0.5 and one of skill 0.9. On the grey line the forecast was exactly right."
_ = plot_forecast_skill(inflows, skills=(0.5, 0.9))

In [ ]:
# @title Код: skill-howto
#| label: skill-howto
#| echo: false
import numpy as np

_, summers = season_totals(inflows)
average_error = {skill: float(np.mean(np.abs(np.asarray(seasonal_forecast(inflows, skill)) - summers))) for skill in (0.0, 0.5, 0.9)}
rosy = {skill: int(np.sum(np.asarray(seasonal_forecast(inflows, skill)) > summers + 10.0)) for skill in (0.5, 0.9)}
gloomy = {skill: int(np.sum(np.asarray(seasonal_forecast(inflows, skill)) < summers - 10.0)) for skill in (0.5, 0.9)}
say(
    f"- **How to read the figure.** One dot per year: across, the summer that came; up, what the forecast said. A dot "
    f"on the grey line is a forecast that was exactly right; the further from the line, the larger the error. With "
    f"skill 0.9 the dots hug the line; with skill 0.5 they form a cloud. The typical error (the average distance "
    f"from the line) is {average_error[0.9]:.0f} Mm³ at skill 0.9 and {average_error[0.5]:.0f} Mm³ at skill 0.5, "
    f"against {average_error[0.0]:.0f} Mm³ for \"an average summer\" every year.\n"
    f"- **A weak forecast stays close to the average.** Look at the skill-0.5 cloud: it is flatter than the line, "
    f"the forecasts hardly ever say 20 or 70 even when the summer turned out so. That is not timidity, it is honesty: "
    f"a forecaster who knows little should say something close to the average, because a bold forecast from weak "
    f"evidence would be wrong more often and by more. The formula above does exactly that: it pulls the guess "
    f"towards the long-run mean in proportion to what the forecaster does not know.\n"
    f"- **Two ways to be wrong.** A forecast that is too rosy says the summer will be fine, the operator gives the "
    f"winter fields their water, and the farm is short in July. One that is too gloomy makes the operator hold back "
    f"water that was not needed. With skill 0.5 the forecast was more than 10 Mm³ too rosy in {rosy[0.5]} of the "
    f"hundred years and more than 10 Mm³ too gloomy in {gloomy[0.5]}; with skill 0.9, in {rosy[0.9]} and "
    f"{gloomy[0.9]}. The two errors are not equally bad for the two users, which is why the operator cannot simply "
    f"take a weak forecast at its word.",
    f"- **Rasmni qanday oʻqish.** Har bir yil — bitta nuqta: koʻndalangiga kelgan yoz, tikka prognoz aytgani. Kulrang "
    f"chiziq ustidagi nuqta — aynan toʻgʻri chiqqan prognoz; chiziqdan qancha uzoq boʻlsa, xato shuncha katta. Sifat "
    f"0,9 da nuqtalar chiziqqa yopishib turadi; 0,5 da bulut hosil qiladi. Odatiy xato (chiziqdan oʻrtacha "
    f"masofa) sifat 0,9 da {average_error[0.9]:.0f} mln m³, 0,5 da {average_error[0.5]:.0f} mln m³, har yili "
    f"\"oʻrtacha yoz\" deyishda esa {average_error[0.0]:.0f} mln m³.\n"
    f"- **Kuchsiz prognoz oʻrtachaga yaqin turadi.** Sifat 0,5 bulutiga qarang: u chiziqdan yotiqroq, yoz 20 yoki 70 "
    f"chiqqanda ham prognoz deyarli hech qachon 20 yoki 70 demaydi. Bu qoʻrqoqlik emas, halollik: kam biladigan "
    f"prognozchi oʻrtachaga yaqin narsa aytishi kerak, chunki kuchsiz dalillardan dadil prognoz tez-tez va koʻproq "
    f"xato boʻladi. Yuqoridagi formula aynan shuni qiladi: taxminni prognozchi bilmagan narsaga mutanosib ravishda "
    f"uzoq muddatli oʻrtacha tomon tortadi.\n"
    f"- **Xato qilishning ikki yoʻli.** Haddan tashqari umidvor prognoz yoz yaxshi boʻladi deydi, operator qishki "
    f"dalalarga suvini beradi, xoʻjalik esa iyulda taqchil qoladi. Haddan tashqari tushkun prognoz operatorni kerak "
    f"boʻlmagan suvni ushlab turishga majbur qiladi. Sifat 0,5 da prognoz yuz yildan {rosy[0.5]} tasida 10 mln m³ "
    f"dan ortiq umidvor, {gloomy[0.5]} tasida 10 mln m³ dan ortiq tushkun boʻldi; sifat 0,9 da — {rosy[0.9]} va "
    f"{gloomy[0.9]} tasida. Bu ikki xato ikki isteʼmolchi uchun bir xil yomon emas, shuning uchun operator kuchsiz "
    f"prognozga shunchaki soʻzsiz ishona olmaydi.",
    f"- **Как читать рисунок.** Одна точка — один год: по горизонтали — какое лето наступило, по вертикали — что "
    f"говорил прогноз. Точка на серой линии — прогноз, который сбылся в точности; чем дальше от линии, тем больше "
    f"ошибка. При качестве 0,9 точки жмутся к линии; при 0,5 образуют облако. Типичная ошибка (среднее расстояние "
    f"от линии) — {average_error[0.9]:.0f} млн м³ при качестве 0,9 и {average_error[0.5]:.0f} млн м³ при 0,5, против "
    f"{average_error[0.0]:.0f} млн м³, если каждый год говорить «среднее лето».\n"
    f"- **Слабый прогноз держится около среднего.** Посмотрите на облако при качестве 0,5: оно положе линии, "
    f"прогноз почти никогда не говорит 20 или 70, даже когда лето оказалось таким. Это не робость, а честность: "
    f"прогнозист, который знает мало, должен говорить что-то близкое к среднему, потому что смелый прогноз по слабым "
    f"данным ошибался бы чаще и сильнее. Формула выше делает ровно это: она притягивает оценку к многолетнему "
    f"среднему пропорционально тому, чего прогнозист не знает.\n"
    f"- **Два способа ошибиться.** Слишком радужный прогноз обещает хорошее лето, оператор отдаёт зимним полям их "
    f"воду, и хозяйству не хватает воды в июле. Слишком мрачный заставляет оператора придержать воду, которая была "
    f"не нужна. При качестве 0,5 прогноз был более чем на 10 млн м³ радужнее истины в {rosy[0.5]} из ста лет и более "
    f"чем на 10 млн м³ мрачнее — в {gloomy[0.5]}; при качестве 0,9 — в {rosy[0.9]} и {gloomy[0.9]}. Эти две ошибки "
    f"не одинаково плохи для двух водопользователей, и потому оператор не может просто верить слабому прогнозу на "
    f"слово.",
)

Теперь оператор пользуется этими прогнозами. Правило не изменилось: оно верит прогнозу на слово и прибавляет его
к воде на горизонте. Меняется план: для каждого качества мы снова перебираем 861 зимний план, так что порог и
доля выбираются с учётом того, насколько хорош прогноз.

In [ ]:
# @title Код: fig-skill-fronts
#| label: fig-skill-fronts
#| fig-cap: "The unbeaten winter plans for forecasts of growing skill. The curves move down as the skill grows, from storage alone to the perfect forecast."
# Six more forecasts of growing skill, 861 winter plans each (this cell runs for a few minutes)
tables = {1.0: with_perfect, **score_by_skill(inflows, (0.0, 0.3, 0.5, 0.65, 0.8, 0.9), winter_grid(n_trigger=41))}
_ = plot_winter_fronts(
    {
        "perfect forecast": with_perfect,
        "forecast with skill 0.8": tables[0.8],
        "forecast with skill 0.5": tables[0.5],
        "storage alone": storage_only,
    }
)

In [ ]:
# @title Код: skill-fronts-text
#| label: skill-fronts-text
#| echo: false
cases = {
    "storage alone": (storage_only, None),
    "skill 0.5": (tables[0.5], seasonal_forecast(inflows, 0.5)),
    "skill 0.8": (tables[0.8], seasonal_forecast(inflows, 0.8)),
    "perfect": (with_perfect, perfect),
}
chosen_plans = {name: winter_plan_within(table, 2.0) for name, (table, _) in cases.items()}
rationed = {
    name: int(np.sum(simulate_winter(inflows, chosen_plans[name], forecast=forecast).winter_delivered_by_year < WINTER_FIELDS_TOTAL_MM3 - 0.01))
    for name, (_, forecast) in cases.items()
}
say(
    f"- **The curves nest.** Each forecast's curve lies between \"storage alone\" and the perfect forecast, and the "
    f"better the forecast, the closer to the perfect one. No curve crosses below the perfect one: a wrong forecast "
    f"can at best be ignored, and the optimizer makes sure it is never worse than that.\n"
    f"- **How a plan copes with a wrong forecast.** Take the plans that hold the farm to 2 Mm³ a year. To reach that "
    f"safety, the plan without a forecast rations {rationed['storage alone']} of the hundred winters; with a forecast "
    f"of skill 0.5 it rations {rationed['skill 0.5']}, with skill 0.8 {rationed['skill 0.8']}, with the perfect "
    f"forecast {rationed['perfect']}. The weaker the forecast, the more winters the plan must hold back to be safe "
    f"against the summers the forecast gets wrong. Those extra rationed winters are the price of the forecast's "
    f"errors, and the optimizer pays it by moving the trigger: {chosen_plans['skill 0.5'].trigger:.0f} Mm³ of water "
    f"in sight with skill 0.5 against {chosen_plans['perfect'].trigger:.0f} with the perfect forecast.\n"
    f"- **The operator does not need to know the formula.** It needs to know the skill, and to choose the plan for "
    f"it. A plan tuned for a perfect forecast and fed a weak one would ration too rarely; that is the trap the next "
    f"figure quantifies.",
    f"- **Egri chiziqlar ichma-ich joylashadi.** Har bir prognozning egri chizigʻi \"faqat ombordagi suv\" bilan "
    f"mukammal prognoz orasida yotadi, va prognoz qancha yaxshi boʻlsa, mukammalga shuncha yaqin. Hech bir egri "
    f"chiziq mukammalnikidan pastga kesib oʻtmaydi: notoʻgʻri prognozni eng yaxshi holda eʼtiborsiz qoldirish "
    f"mumkin, optimallashtirgich esa undan yomon boʻlmasligini taʼminlaydi.\n"
    f"- **Reja notoʻgʻri prognoz bilan qanday kurashadi.** Xoʻjalikni yiliga 2 mln m³ da ushlab turadigan rejalarni "
    f"oling. Shu xavfsizlikka erishish uchun prognozsiz reja yuz qishdan {rationed['storage alone']} tasini "
    f"cheklaydi; sifati 0,5 prognoz bilan {rationed['skill 0.5']} tasini, 0,8 bilan {rationed['skill 0.8']} tasini, "
    f"mukammal prognoz bilan {rationed['perfect']} tasini. Prognoz qancha kuchsiz boʻlsa, reja prognoz xato "
    f"qiladigan yozlardan xavfsiz boʻlish uchun shuncha koʻp qishda ushlab turishi kerak. Bu qoʻshimcha cheklangan "
    f"qishlar — prognoz xatolarining narxi, optimallashtirgich uni chegarani surish bilan toʻlaydi: sifat 0,5 da "
    f"koʻrinib turgan suv {chosen_plans['skill 0.5'].trigger:.0f} mln m³, mukammal prognozda "
    f"{chosen_plans['perfect'].trigger:.0f}.\n"
    f"- **Operator formulani bilishi shart emas.** U sifatni bilishi va unga mos rejani tanlashi kerak. Mukammal "
    f"prognozga moslangan, lekin kuchsiz prognoz berilgan reja juda kam cheklagan boʻlardi; keyingi rasm aynan shu "
    f"tuzoqni oʻlchaydi.",
    f"- **Кривые вложены друг в друга.** Кривая каждого прогноза лежит между «только запасом» и идеальным "
    f"прогнозом, и чем лучше прогноз, тем ближе к идеальному. Ни одна кривая не опускается ниже идеальной: ошибочный "
    f"прогноз в лучшем случае можно проигнорировать, и оптимизатор следит, чтобы хуже этого не стало.\n"
    f"- **Как план справляется с ошибочным прогнозом.** Возьмём планы, удерживающие хозяйство на 2 млн м³ в год. "
    f"Чтобы достичь этой безопасности, план без прогноза лимитирует {rationed['storage alone']} из ста зим; с "
    f"прогнозом качества 0,5 — {rationed['skill 0.5']}, с качеством 0,8 — {rationed['skill 0.8']}, с идеальным "
    f"прогнозом — {rationed['perfect']}. Чем слабее прогноз, тем больше зим план должен придерживать воду, чтобы "
    f"застраховаться от тех лет, где прогноз ошибается. Эти лишние лимитированные зимы — цена ошибок прогноза, и "
    f"оптимизатор платит её сдвигом порога: {chosen_plans['skill 0.5'].trigger:.0f} млн м³ воды на горизонте при "
    f"качестве 0,5 против {chosen_plans['perfect'].trigger:.0f} при идеальном прогнозе.\n"
    f"- **Оператору не нужно знать формулу.** Ему нужно знать качество и выбрать план под него. План, настроенный "
    f"на идеальный прогноз и получающий слабый, лимитировал бы слишком редко; именно эту ловушку измеряет следующий "
    f"рисунок.",
)

**Что мы находим, на одном графике.** Три кривые трудно сравнить на глаз, поэтому сводим каждую к одному числу.
Фиксируем безопасность хозяйства на летнем дефиците 2 млн м³ в год — точке, которой мы пользовались всё время, —
и спрашиваем: с этим прогнозом сколько должны отдать зимние поля, чтобы хозяйство оставалось настолько в
безопасности? Одна точка на каждое качество, от «среднего лета каждый год» до идеального прогноза, а оператор без
прогноза — линия, которую нужно превзойти.

In [ ]:
# @title Код: fig-forecast-value
#| label: fig-forecast-value
#| fig-cap: "What the forecast is worth against its skill: the winter shortage left when the farm's shortage is held to 2 Mm³ a year. The orange line is the operator without a forecast."
curve = forecast_value(tables, summer_cap=2.0)
_ = plot_forecast_value(curve, no_forecast=at_2, summer_cap=2.0)

In [ ]:
# @title Код: forecast-value-text
#| label: forecast-value-text
#| echo: false
value_at = dict(zip(curve["skill"], curve["winter_shortage"]))
half_way = (at_2 - value_at[0.5]) / (at_2 - value_at[1.0]) * 100.0
say(
    f"- **A forecast with no skill is no forecast.** At skill 0 the forecast says 40 Mm³ every year, and the rule is "
    f"back to watching the storage: the curve starts on the orange line, within the grid's steps "
    f"({value_at[0.0]:.1f} against {at_2:.1f} Mm³ a year).\n"
    f"- **Value grows with skill, but not in proportion.** A forecast of skill 0.5, halfway to perfect, delivers "
    f"{half_way:.0f} % of what the perfect forecast delivers; one of skill 0.8 leaves the winter fields "
    f"{value_at[0.8]:.1f} Mm³ a year short, against {value_at[1.0]:.1f} for the perfect forecast and {at_2:.1f} "
    f"with none. Most of the value sits in the last stretch of skill. The curve wiggles because the forecast "
    f"errors are one random draw of a hundred years; the trend is what counts.\n"
    f"- **Where real forecasts sit.** The statistical forecasts that Apel et al. (2018) built for rivers of Central "
    f"Asia explain 60 to 80 % of the year-to-year variation of the April–September flow when issued in January, "
    f"and 68 to 97 % when issued on 1 April, the date that matters most for the region's water planning. In our "
    f"terms that is a skill of roughly 0.8 to 0.95: real forecasts sit on the right-hand part of the curve, where "
    f"the value is.\n"
    f"- **The forecast's worth depends on the decision, not only on the forecast.** The same forecast was worth "
    f"nothing for the summer decisions of this reservoir (we tried: the operator learns the summer by May, and "
    f"water saved in one month is still there the next), and worth {at_2 - value_at[1.0]:.1f} Mm³ a year for the "
    f"winter decision. A forecast is worth what the decision it informs can do with it, a point made for weather "
    f"forecasts by Murphy (1993) and for reservoirs by Anghileri et al. (2016).\n"
    f"- **What the operator does with a wrong forecast matters.** Here the rule takes the forecast at its word, and "
    f"the optimizer compensates by choosing the trigger. A cautious operator could count only part of the "
    f"forecast; exercise 5 asks whether that helps.\n"
    f"- **What a real forecast service adds.** Not one number but a range: an ensemble of possible summers, or a "
    f"best guess with an error band, and updates every month as the snow accumulates. The natural next step, not "
    f"taken here, is to count on a cautious quantile of that range (\"the summer we are 80 % sure of\") and let the "
    f"optimizer choose how cautious; and to let the forecast improve through the winter, as exercise 6 begins to "
    f"do.",
    f"- **Sifati yoʻq prognoz — prognoz emas.** Sifat 0 da prognoz har yili 40 mln m³ deydi, va qoida yana "
    f"ombordagi suvga qarashga qaytadi: egri chiziq toʻr qadamlari doirasida toʻq sariq chiziqdan boshlanadi "
    f"(yiliga {value_at[0.0]:.1f} ga qarshi {at_2:.1f} mln m³).\n"
    f"- **Qadr sifat bilan oʻsadi, lekin mutanosib emas.** Sifati 0,5 boʻlgan, mukammalga yarim yoʻl bosgan "
    f"prognoz mukammal prognoz beradiganning {half_way:.0f} % ini beradi; sifati 0,8 boʻlgani qishki dalalarni "
    f"yiliga {value_at[0.8]:.1f} mln m³ taqchil qoldiradi, mukammal prognozda {value_at[1.0]:.1f}, prognozsiz "
    f"{at_2:.1f}. Qadrning koʻp qismi sifatning oxirgi boʻlagida yotadi. Egri chiziq tebranadi, chunki prognoz "
    f"xatolari — yuz yillik bitta tasodifiy tanlanma; muhimi — umumiy yoʻnalish.\n"
    f"- **Haqiqiy prognozlar qayerda turadi.** Apel va boshq. (2018) Markaziy Osiyo daryolari uchun qurgan statistik "
    f"prognozlar aprel–sentyabr oqimining yildan yilga oʻzgarishining 60–80 % ini yanvarda berilganda, 68–97 % ini "
    f"esa 1 aprelda — mintaqa suv rejalashtiruvi uchun eng muhim sanada — berilganda tushuntiradi. Bizning "
    f"atamalarda bu taxminan 0,8 dan 0,95 gacha sifat: haqiqiy prognozlar egri chiziqning oʻng qismida, qadr "
    f"yotgan joyda turadi.\n"
    f"- **Prognozning qadri faqat prognozga emas, qarorga bogʻliq.** Xuddi shu prognoz bu omborning yozgi "
    f"qarorlari uchun hech narsaga arzimadi (biz sinab koʻrdik: operator yozni may oyigacha bilib oladi, bir oyda "
    f"saqlangan suv esa keyingi oyda ham joyida turadi), qishki qaror uchun esa yiliga "
    f"{at_2 - value_at[1.0]:.1f} mln m³ ga arzidi. Prognoz u xabardor qilgan qaror u bilan nima qila olsa, "
    f"shunchaga arziydi — bu fikrni ob-havo prognozlari uchun Murphy (1993), suv omborlari uchun Anghileri va "
    f"boshq. (2016) aytgan.\n"
    f"- **Operator notoʻgʻri prognoz bilan nima qilishi muhim.** Bu yerda qoida prognozga soʻzsiz ishonadi, "
    f"optimallashtirgich esa chegarani tanlab buni qoplaydi. Ehtiyotkor operator prognozning faqat bir qismini "
    f"hisobga olishi mumkin edi; 5-mashq bu yordam beradimi, deb soʻraydi.\n"
    f"- **Haqiqiy prognoz xizmati nima qoʻshadi.** Bitta raqam emas, oraliq: mumkin boʻlgan yozlar ansambli yoki xato "
    f"oraligʻi bilan eng yaxshi taxmin, va qor toʻplangani sari har oy yangilanish. Bu yerda qilinmagan tabiiy "
    f"keyingi qadam — shu oraliqning ehtiyotkor kvantiliga (\"80 % ishonchimiz komil boʻlgan yoz\") tayanish va "
    f"qanchalik ehtiyotkor boʻlishni optimallashtirgichga tanlatish; hamda prognozni qish davomida yaxshilanib "
    f"borishiga yoʻl qoʻyish, 6-mashq buni boshlaydi.",
    f"- **Прогноз без качества — не прогноз.** При качестве 0 прогноз каждый год говорит 40 млн м³, и правило снова "
    f"смотрит только на запас: кривая начинается на оранжевой линии, в пределах шага сетки "
    f"({value_at[0.0]:.1f} против {at_2:.1f} млн м³ в год).\n"
    f"- **Ценность растёт с качеством, но не пропорционально.** Прогноз качества 0,5, на полпути к идеальному, даёт "
    f"{half_way:.0f} % того, что даёт идеальный; прогноз качества 0,8 оставляет зимним полям дефицит "
    f"{value_at[0.8]:.1f} млн м³ в год против {value_at[1.0]:.1f} у идеального и {at_2:.1f} без прогноза. "
    f"Большая часть ценности лежит на последнем отрезке качества. Кривая колеблется, потому что ошибки прогноза — "
    f"одна случайная выборка из ста лет; важна тенденция.\n"
    f"- **Где находятся настоящие прогнозы.** Статистические прогнозы, которые Apel et al. (2018) построили для рек "
    f"Центральной Азии, объясняют 60–80 % межгодовой изменчивости стока за апрель–сентябрь при выпуске в январе и "
    f"68–97 % при выпуске 1 апреля — в дату, которая важнее всего для водного планирования региона. В наших "
    f"терминах это качество примерно от 0,8 до 0,95: настоящие прогнозы находятся на правой части кривой, там, "
    f"где и лежит ценность.\n"
    f"- **Ценность прогноза зависит от решения, а не только от прогноза.** Тот же прогноз ничего не стоил для "
    f"летних решений этого водохранилища (мы проверяли: оператор узнаёт лето к маю, а вода, сбережённая в одном "
    f"месяце, никуда не девается к следующему) и стоил {at_2 - value_at[1.0]:.1f} млн м³ в год для зимнего "
    f"решения. Прогноз стоит столько, сколько может сделать с ним решение, которое он обслуживает, — мысль, "
    f"высказанная для прогнозов погоды Murphy (1993), а для водохранилищ — Anghileri et al. (2016).\n"
    f"- **Важно, что оператор делает с ошибочным прогнозом.** Здесь правило верит прогнозу на слово, а "
    f"оптимизатор компенсирует это выбором порога. Осторожный оператор мог бы учитывать лишь часть прогноза; "
    f"упражнение 5 спрашивает, помогает ли это.\n"
    f"- **Что добавляет настоящая прогностическая служба.** Не одно число, а диапазон: ансамбль возможных лет или "
    f"лучшую оценку с полосой ошибки, и обновления каждый месяц по мере накопления снега. Естественный следующий "
    f"шаг, здесь не сделанный, — опираться на осторожный квантиль этого диапазона («лето, в котором мы уверены на "
    f"80 %») и дать оптимизатору выбрать меру осторожности; и позволить прогнозу уточняться по ходу зимы, с чего "
    f"начинает упражнение 6.",
)

**Одной фразой.** На прогноз, который иногда ошибается, всё же стоит опираться, но оператор должен делать это по
плану, выбранному под его качество; чем лучше прогноз, тем больше он стоит, и большая часть ценности лежит на
последнем отрезке между хорошим прогнозом и идеальным — ровно там, где находятся настоящие прогнозы
гидрометеорологических служб региона, и потому их качество важно для такого водохранилища.

# Другая река: режим из статьи

Река из статьи многоводна зимой и маловодна летом — противоположность нашей. То же водохранилище, то же
хозяйство, те же сто случайных значений; только сезоны поменяны местами. Выглядит ли компромисс так же?

In [ ]:
# @title Код: fig-regimes
#| label: fig-regimes
#| fig-cap: "Left: the average month of both rivers against the farm's need. Right: the unbeaten plans of both. Same reservoir, same need, same random seed."
paper_inflows = synthetic_inflows(regime=PAPER_REGIME)  # the paper's seasons: winter 40 Mm³, summer 25 Mm³
paper_table = score_plans(paper_inflows, plan_grid())  # the same 441 plans on that river

_ = plot_regime_comparison(
    {"Central Asian regime (this notebook)": (inflows, table), "Paper regime (winter peak)": (paper_inflows, paper_table)}
)

In [ ]:
# @title Код: regimes-text
#| label: regimes-text
#| echo: false
paper_front = front_of(paper_table)
paper_base = simulate_plan(paper_inflows, NO_RATIONING).scores
ours_50 = front[front["worst_month"] <= 50.0].iloc[0]
paper_50 = paper_front[paper_front["worst_month"] <= 50.0].iloc[0]
say(
    f"- **How to read the figure.** Left: the average inflow of each month, blue for our river, orange for the "
    f"paper's; the black step is the farm's need, 0.83 Mm³ a month in winter and 7.5 in summer. Our summer bars "
    f"reach almost up to the need; the paper's summer bars stay far below it, so its reservoir has to carry most "
    f"of the summer water over from winter. Right: the unbeaten plans of both rivers, on the chart you know.\n"
    f"- **Same shape.** Both curves bend the same way: cheap at first, expensive at the end. The example works for "
    f"either river.\n"
    f"- **The paper's river is the harder case.** Without rationing it is {paper_base.worst_month:.0f} % short in "
    f"its worst month (ours: {s.worst_month:.0f} %), and holding the worst month to 50 % costs "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} Mm³ per year there against "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} here. With a summer peak the water arrives when the farm "
    f"needs it, and the reservoir only has to cover the gap.\n"
    f"- **What runs our reservoir dry** is a poor winter (little to store) followed by a weak melt. That is the "
    f"Central Asian pattern: the reservoir is filled from the low winter flow, as Kattakurgan is.",
    f"- **Rasmni qanday oʻqish.** Chapda: har bir oyning oʻrtacha oqimi, koʻk — bizning daryo, toʻq sariq — "
    f"maqoladagi daryo; qora pogʻona — xoʻjalik talabi, qishda oyiga 0,83 mln m³, yozda 7,5. Bizning yoz ustunlari "
    f"deyarli talabgacha yetadi; maqoladagi daryoning yoz ustunlari undan ancha pastda, shuning uchun uning ombori "
    f"yoz suvining koʻp qismini qishdan olib oʻtishi kerak. Oʻngda: ikkala daryoning yengilmagan rejalari, sizga "
    f"tanish diagrammada.\n"
    f"- **Shakli bir xil.** Ikkala egri chiziq ham bir xil bukiladi: avval arzon, oxirida qimmat. Misol ikkala daryo "
    f"uchun ham ishlaydi.\n"
    f"- **Maqoladagi daryo — qiyinroq holat.** Cheklashsiz uning eng yomon oyi {paper_base.worst_month:.0f} % "
    f"taqchil (bizniki: {s.worst_month:.0f} %), va eng yomon oyni 50 % da ushlab turish u yerda yiliga "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} mln m³, bizda esa "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} mln m³ ga tushadi. Yozgi toshqin bilan suv xoʻjalikka kerak "
    f"paytda keladi, va ombor faqat farqni qoplashi kerak.\n"
    f"- **Bizning omborni quritadigan narsa** — kam suvli qish (saqlashga kam suv) va undan keyingi kuchsiz erish. "
    f"Bu Markaziy Osiyo manzarasi: ombor, Kattaqoʻrgʻon suv ombori kabi, kam suvli qishki oqimdan toʻldiriladi.",
    f"- **Как читать рисунок.** Слева: средний приток каждого месяца, синий — наша река, оранжевый — река из "
    f"статьи; чёрная ступенька — потребность хозяйства, 0,83 млн м³ в месяц зимой и 7,5 летом. Наши летние столбики "
    f"почти дотягивают до потребности; летние столбики реки из статьи остаются далеко ниже, так что её "
    f"водохранилищу приходится переносить большую часть летней воды с зимы. Справа: непревзойдённые планы обеих "
    f"рек на уже знакомом графике.\n"
    f"- **Та же форма.** Обе кривые изгибаются одинаково: сначала дёшево, в конце дорого. Пример работает для обеих "
    f"рек.\n"
    f"- **Река из статьи — более трудный случай.** Без лимитирования её худший месяц имеет дефицит "
    f"{paper_base.worst_month:.0f} % (у нашей: {s.worst_month:.0f} %), а удержание худшего месяца на уровне 50 % "
    f"стоит там {paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} млн м³ в год против "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} у нас. При летнем пике вода приходит тогда, когда она нужна "
    f"хозяйству, и водохранилищу остаётся лишь покрыть разницу.\n"
    f"- **Наше водохранилище опустошает** скудная зима (мало что накопить), за которой следует слабое таяние. Это "
    f"центральноазиатская картина: водохранилище наполняется из скудного зимнего стока, как Каттакурганское.",
)

# От игрушечной модели к настоящему водохранилищу

- **У настоящих правил те же рычаги, только много раз.** В правиле сброса Каттакурганского водохранилища в модели
  Зарафшана из этого репозитория (`ZRBReleaseRule`) есть уровень наполнения, ниже которого сброс урезается (`v1`),
  и коэффициент урезания (`buffer_coef`): наши порог и доля, с отдельным значением для каждого месяца. Поэтому
  полной модели нужен оптимизатор.
- **Чего эта игрушечная модель не учитывает.** Прогноз, который уточняется по ходу зимы, и полосу неопределённости,
  которая к нему прилагается; больше водопользователей с разными приоритетами; как на самом деле культуры
  реагируют на дефицит; испарение, зависящее от площади зеркала; возвратные воды.
- **Одна река — это одна река.** Числа здесь относятся к этому единственному синтетическому ряду в сто лет. Другие
  ряды дают другие числа. Форма (изогнутая кривая от «без лимитирования» до «сильного урезания») была той же для
  трёх других рядов, которые мы проверили.

# Упражнения

1. **Водохранилище побольше.** В ячейке настроек в начале тетради задайте `capacity=60.0` и запустите тетрадь
   заново. Окупается ли лимитирование по-прежнему? Где теперь изгиб?
2. **Другая вторая цель.** Замените «худший месяц» на «дефицитные годы» в `objectives()` и снова запустите
   оптимизатор. Какие планы теперь непревзойдённые? Остаётся ли среди них «без лимитирования»?
3. **Другая река.** Используйте `synthetic_inflows(seed=1)`, затем `synthetic_inflows(regime=PAPER_REGIME, seed=1)`.
   Какие числа меняются, а что остаётся прежним?
4. **Сезонное правило.** Дайте правилу две доли: одну для начала лета, другую для его конца. Найдёт ли оптимизатор
   планы, превосходящие кривую с двумя рычагами?
5. **Осторожный оператор.** В `WinterRelease` учитывайте только часть прогноза: `node.storage + 0.7 * forecast`.
   Пересчитайте кривую качества. Помогает ли осторожность при слабом прогнозе (качество 0,5)? Вредит ли при
   сильном?
6. **Поздний прогноз.** Настоящие прогнозы приходят в конце зимы, когда снег уже выпал. Пусть правило использует
   прогноз только в последние два зимних месяца (`month >= 4`), а до того смотрит только на запас. Какая часть
   ценности прогноза сохраняется?

Как выполнить упражнение в Colab: код каждой ячейки свёрнут; нажмите «Show code», чтобы увидеть и изменить его, а
затем снова запустите ячейки ниже. Для упражнения 1 измените число в ячейке настроек в начале тетради и запустите
тетрадь заново оттуда.

# Источники

- Hashimoto, T., Stedinger, J. R., Loucks, D. P. (1982). Reliability, resiliency, and vulnerability criteria for
  water resource system performance evaluation. *Water Resources Research* 18(1), 14–20. The reservoir, its
  numbers (p. 17, Table 1; seasons swapped here) and the trade-off between failing often and failing badly
  (Fig. 9).
- Draper, A. J., Lund, J. R. (2004). Optimal hedging and carryover storage value. *Journal of Water Resources
  Planning and Management* 130(1), 83–87. When rationing pays (p. 83) and why a rule with two parameters is often
  enough (p. 87).
- Apel, H., Abdykerimova, Z., Agalhanova, M., Baimaganbetov, A., Gavrilenko, N., Gerlitz, L., Kalashnikova, O.,
  Unger-Shayesteh, K., Vorogushyn, S., Gafurov, A. (2018). Statistical forecast of seasonal discharge in Central
  Asia using observational records: development of a generic linear modelling tool for operational water resource
  management. *Hydrology and Earth System Sciences* 22, 2225–2254. How the region's hydrometeorological services
  forecast the April–September flow, and from what.
- Anghileri, D., Voisin, N., Castelletti, A., Pianosi, F., Nijssen, B., Lettenmaier, D. P. (2016). Value of
  long-term streamflow forecasts to reservoir operations for water supply in snow-dominated river catchments.
  *Water Resources Research* 52, 4209–4225. Forecast value measured against the perfect forecast, for a reservoir.
- Murphy, A. H. (1993). What is a good forecast? An essay on the nature of goodness in weather forecasting.
  *Weather and Forecasting* 8, 281–293. The distinction between a forecast's quality and its value to a decision.
- Deb, K., Pratap, A., Agarwal, S., Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm:
  NSGA-II. *IEEE Transactions on Evolutionary Computation* 6(2), 182–197. The search method behind
  `taqsim.optimize`.